# Augusto's adaptive-variance sKFs beside the banks

Augusto's last commits on `time_varying_system` (2026-10-06 to 2026-10-08, up to `6a27d96`) add five sKFs that set their
variance by a rule of their own instead of a fixed $\varepsilon$: AWU-sKF v1 and v2, AWU-sKF-L v1 and v2, and GISV-sKF.
His notebook `time_varying_systems.ipynb` runs them in his setup ($L = 16$, a sinc response, white input, about 30 dB)
and has no saved outputs. Here they run in our setup, notebook 09's change test, beside the filters of
`kf-bank-q-sweep.ipynb` (commit `56108a2`): the exact sKF, the exact fKF and the KF at $-20$ dB, and the banks of sKFs,
fKFs and KFs.

**The five filters.** Each takes the sKF step, $\boldsymbol{w}_t = \boldsymbol{w}_{t-1} + \boldsymbol{x}_t\,\tilde v_t e_t/S_t$ and
$v_t = \tilde v_t\big(1 - \tilde v_t\lVert\boldsymbol{x}_t\rVert^2/(M S_t)\big)$, with
$S_t = v_\eta + \tilde v_t\lVert\boldsymbol{x}_t\rVert^2$ for the Gaussian likelihood or
$S_t = b_\eta\lvert e_t\rvert + \tilde v_t\lVert\boldsymbol{x}_t\rVert^2$ for the minorized Laplacian one (eq. 45). Only
$\tilde v_t$ differs from the sKF's $v_{t-1} + \varepsilon$. With $J_t = v_\eta + v_{t-1}\lVert\boldsymbol{x}_t\rVert^2$:

| filter | likelihood | $\tilde v_t$ |
|---|---|---|
| AWU-sKF v1 | Gaussian | $v_{t-1}\big(1 + v_{t-1}\lVert\boldsymbol{x}_t\rVert^2(e_t^2 - J_t)/J_t^2\big)$ |
| AWU-sKF v2 | Gaussian | $v_{t-1}\big(1 + v_{t-1}\lVert\boldsymbol{x}_t\rVert^2 e_t^2/(M J_t^2)\big)$ |
| GISV-sKF | Gaussian | $v_{t-1} + \lVert\boldsymbol{w}_{t-1} - \boldsymbol{w}_{t-2}\rVert^2/M$ |
| AWU-sKF-L v1, v2 | minorized Laplacian | as AWU-sKF v1, v2, with $v_\eta$ replaced by $b_\eta\lvert e_t\rvert$ in $J_t$ |

- AWU-sKF v1 is a gradient step in $v$ on the Gaussian log density of $e_t$, with step $2v_{t-1}^2$:
  $\partial\log\mathcal{N}(e_t; 0, J_t)/\partial v = \lVert\boldsymbol{x}_t\rVert^2(e_t^2 - J_t)/(2J_t^2)$. It moves $v$ up or
  down.
- AWU-sKF v2 and GISV-sKF add the same amount, $\lVert\boldsymbol{x}_t\rVert^2 g_t^2/M$ with
  $g_t = v e_t/(v_\eta + v\lVert\boldsymbol{x}_t\rVert^2)$, the squared size of an sKF step over $M$; v2 adds it before the
  update of step $t$, GISV after it (for step $t+1$). GISV-sKF is the estimator of the draft's Remark 5,
  $\hat\varepsilon_t = \lVert\boldsymbol{w}_t - \boldsymbol{w}_{t-1}\rVert^2/M$ (eq. 76 of [5]), listed in Remark 6 as
  "eq. (76) of [5]".
- The AWU rules use $e_t$ itself, so their $\tilde v_t$ already depends on $y_t$ and is not a prior for step $t$.
  GISV-sKF uses only the past.

**Choices made with Ramiro before writing code (2026-10-13)**

| doubt | choice |
|---|---|
| scenario | ours: notebook 09's change test in the four scenarios of `kf-bank-q-sweep.ipynb`, AR($-0.9$) and white input, $\beta^* = 0.2$ and 2, with the same signals and seeds |
| our filters | the exact sKF, exact fKF and KF at $-20$ dB and the three banks, read from `results/kf-bank-q-sweep/`, where `kf-bank-q-sweep.ipynb` saved its runs. The copied code recomputes a run only if its file is missing |
| banks' $q$ | $10^{-4}$, as in every earlier comparison, and each bank's best $q$ of the sweep (the lowest mean over the record, from the mean curves) |
| Augusto's filters | all five, copied verbatim from `filters.py` at `6a27d96`, in every scenario. $v_\eta$ = the true noise variance for the Gaussian ones, $b_\eta = \mathrm{E}\lvert\eta_t\rvert$ for the Laplacian ones (his notebook also gives them the true values), $v_0 = 2$ as ours (his notebook: 1) |
| base filters | the filter each rule modifies, with a fixed $\varepsilon$ tuned to $-20$ dB by notebook 09's search: his `sKF_algorithm` (Gaussian) and `sKF_L_algorithm` (minorized Laplacian). Each rule is then measured against the same filter with a fixed $\varepsilon$, apart from the likelihood |
| measures | those of `kf-bank-q-sweep.ipynb`: misalignment and excess error, from the mean and from the median over the realisations; and $v_t$ of his filters and of the base filters |
| where | this new notebook. His and the base filters' runs are saved under `results/awu-gisv-vs-banks/` (git-ignored) as they finish |

**Run time** 3.0 min on 2026-10-13, 1.7 min of it the runs of Augusto's and the base filters, with the runs of ours read
from `results/kf-bank-q-sweep/`. Run again with `results/awu-gisv-vs-banks/` in place, about a minute. Without
`results/kf-bank-q-sweep/`, the copied code recomputes the runs of ours used here: about 2 h more, from the run times
saved there.

## 1. Imports

In [ ]:
import pickle
import time
from collections import namedtuple
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from scipy.ndimage import uniform_filter1d
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr, logsumexp
from scipy.stats import gennorm
import numba as nb
from numba import njit
import rir_generator as rir

%config InlineBackend.figure_format = 'retina'
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Section 2 of `kf-bank-q-sweep.ipynb` (commit `56108a2`), nothing edited: notebook 09's constants and signals, with the AR
coefficient and the noise shape `beta` as parameters, and `clean_output`, $\boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t$
for every realisation.

In [ ]:
# === copied from kf-bank-q-sweep.ipynb section 2 (commit 56108a2), NOT edited ===
# === copied from bank-kf-excess-error.ipynb section 2 (commit f2ded06), which copied it from bank-skf-fkf.ipynb
#     (57bd568); one option added: the noise shape beta, default BETA (the same draws as before) ===
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run at rest; the change run has 2N
R = 20                      # independent realisations
WARMUP = 500                # AR samples discarded so the input starts stationary
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
SNR_DB = 5.0                # nominal signal-to-noise ratio
VAR_THETA_0 = 2.0           # initial prior variance on each weight (ours; Leszek's is 1/M)
FS = 8000                   # sampling rate [Hz]
TARGET_DB = -20.0           # floor the single filters are tuned to
BETA_GAUSS = 2.0            # the generalized Gaussian with beta = 2 is the Gaussian
DRIFT_DB = 0.5              # settling rule of notebook 09

# === copied from leszek-setup.ipynb section 4 (commit adb148b), NOT edited ===
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]
ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM, reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)
lags = np.abs(np.subtract.outer(np.arange(M), np.arange(M)))


def noise_of(ar_a, beta=BETA):
    # Noise scale and b_eta = E|eta| at 5 dB for this input: notebook 09's lines, AR as a parameter.
    P_signal = ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho
    var_eta = P_signal/10**(SNR_DB/10)
    scale_gg = np.sqrt(var_eta/np.exp(gammaln(3/beta) - gammaln(1/beta)))
    return scale_gg, scale_gg*np.exp(gammaln(2/beta) - gammaln(1/beta))


def signals(seed, ar_a, n, change_at=None, beta=BETA):
    # generate_signals / generate_signals_change of notebook 09 (commit 6e6dabc), with the AR
    # coefficient, the length and the instant of the flip as parameters. Same draws, same order.
    scale_gg, _ = noise_of(ar_a, beta)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(n + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    y = np.convolve(ho, x)[:n]
    if change_at is not None:
        y[change_at:] = -y[change_at:]
    eta = gennorm.rvs(beta, scale=scale_gg, size=n, random_state=rng)
    return x, y + eta
# === end of the copied block ===


def noise_variance(ar_a, beta=BETA):
    # The true noise variance, for the Gaussian likelihood: the variance of the generalized Gaussian.
    scale_gg, _ = noise_of(ar_a, beta)
    return scale_gg**2*np.exp(gammaln(3/beta) - gammaln(1/beta))


def stack(sigs):
    return np.array([s[0] for s in sigs]), np.array([s[1] for s in sigs])


for beta, ar_a in ((BETA, -0.9), (BETA, 0.0), (BETA_GAUSS, -0.9), (BETA_GAUSS, 0.0)):
    scale_gg, b_eta = noise_of(ar_a, beta)
    print(f"beta* = {beta:g}, AR {ar_a:>4}: var_eta = {noise_variance(ar_a, beta):.4e}, "
          f"noise scale = {scale_gg:.4e}, "
          f"b_eta = E|eta| = {b_eta:.4e}")
# === end of the copied block ===


def clean_output(X, change_at=None):
    # x_t' theta_t for every row of X, as signals() forms y: theta = h_o, and -h_o from change_at on
    Y = np.array([np.convolve(ho, x)[:X.shape[1]] for x in X])
    if change_at is not None:
        Y[:, change_at:] = -Y[:, change_at:]
    return Y


def signal_power(ar_a):
    # h_o' R_xx h_o: the line of noise_of
    return ho @ (ar_a**lags if ar_a != 0 else np.eye(M)) @ ho


# Copied from bank-skf-fkf.ipynb section 7 (commit 57bd568).
def scenario_signals(ar_a, beta=BETA):
    return (stack([signals(seed, ar_a, N, beta=beta) for seed in range(R)]),
            stack([signals(seed, ar_a, 2*N, change_at=N, beta=beta) for seed in range(R)]))

## 3. The filters

### 3.1 Ours

Sections 3 and 3.1 of `kf-bank-q-sweep.ipynb` (commit `56108a2`), nothing edited: `sKF_batch`, `fKF_batch`, `KF_batch`,
the sKF and fKF banks (`bank_batch`) and the bank of KFs (`kf_bank_batch`, Algorithm 3). Here they run only for a run of
ours missing from `results/kf-bank-q-sweep/`, for the checks of section 4, and (`sKF_batch`) for the base filters'
search. Below them `chi_minorized` from `fkf.ipynb` (commit `058a9f9`), for that search.

In [ ]:
# === copied from kf-bank-q-sweep.ipynb section 3 (commit 56108a2), NOT edited ===
# === copied from bank-kf-excess-error.ipynb section 3 (commit f2ded06), NOT edited; log_density_gaussian added at the end ===
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/07_skf_conjunto.ipynb, commit 9406077,
#         via bank-skf-fkf.ipynb (commit 57bd568)
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


# Copied from bank-skf-fkf.ipynb (commit 57bd568), from vkf-kf.ipynb.
def chi_gaussian(e, sigma, v_eta):
    """Eq. (38), the Gaussian instance. Used only for the Kalman cross-check of section 4."""
    frac = sigma**2/(sigma**2 + v_eta)
    return frac*e, frac


# sKF_batch and fKF_batch: copied from bank-skf-fkf.ipynb (commit 57bd568), where they carry the options below;
# one option added here, `excess`, and nothing else changed.
#   flip_at: misalignment against -h from that step on (fkf.ipynb had it for the fKF only)
#   clip:    chi' -> max(chi', 0) in the variance update, eq. (36); the mean update is untouched
#   record:  also return chi'_t and v_t at every step
#   v0:      initial v (VAR_THETA_0 = 2 in every notebook of ours; 1/M in Leszek's ggbench.py)
#   wait:    first step that updates (M: wait for a full window, as every notebook of ours)
#   excess:  also return s_t = x_t'(theta_t - w_{t-1}) at every step (not with record)
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn, flip_at=None, clip=False, record=False,
              v0=VAR_THETA_0, wait=None, excess=False):
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(v0))
    eps = np.asarray(epsilon, dtype=float)
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    if record:
        slope_hist, v_hist = np.full((B, n), np.nan), np.full((B, n), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < first:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - (np.maximum(slope, 0.0) if clip else slope)/L)
        if record:
            slope_hist[:, t], v_hist[:, t] = slope, v
    if excess:
        return mis, s
    if record:
        return mis, slope_hist, v_hist
    return mis


def fKF_batch(X, D, h, b_eta, v, chi_fn, flip_at=None, wait=None, excess=False):
    """B fKFs of eq. (37) side by side. Misalignment against h, or against -h from step flip_at on."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.asarray(v, dtype=float)
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < first:
            continue
        power = np.einsum("bm,bm->b", x_t, x_t)
        chi, _ = chi_fn(e, np.sqrt(v*power), b_eta)
        w = w + x_t*(chi/power)[:, None]
    if excess:
        return mis, s
    return mis


# KF_batch: copied from vkf-kf.ipynb (commit 058a9f9); options flip_at and excess added (as in sKF_batch above),
# nothing else changed.
def KF_batch(X, D, h, b_eta, epsilon, chi_fn=chi_laplacian, flip_at=None, excess=False):
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    Sigma = np.broadcast_to(np.eye(L)*float(VAR_THETA_0), (B, L, L)).copy()
    eps_I = np.asarray(epsilon, dtype=float)[:, None, None]*np.eye(L)
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    if excess:
        s = np.empty((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = (w - h) if flip_at is None or t < flip_at else (w + h)
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_{t-1})
        if t < L:
            continue
        St = Sigma + eps_I
        Sx = np.einsum("bij,bj->bi", St, x_t)
        sigma2 = np.einsum("bi,bi->b", x_t, Sx)
        sigma = np.sqrt(sigma2)
        k_hist[:, t] = sigma/b_eta
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + Sx*(chi/sigma2)[:, None]
        Sigma = St - (slope/sigma2)[:, None, None]*np.einsum("bi,bj->bij", Sx, Sx)
        Sigma = 0.5*(Sigma + np.swapaxes(Sigma, 1, 2))
    if excess:
        return mis, k_hist, s
    return mis, k_hist


# === copied from bank-skf-fkf.ipynb section 4 (commit 57bd568); bank_batch has the option excess added ===
LOG_2PI = np.log(2*np.pi)


def transition_matrix(K, q):
    """p_ij of eq. (eps.chain): q to each neighbour on the grid, 1 - q n_i to stay."""
    P = np.zeros((K, K))
    for i in range(K):
        neighbours = [j for j in (i - 1, i + 1) if 0 <= j < K]   # n_i = 1 at the ends, 2 inside
        P[i, neighbours] = q
        P[i, i] = 1 - q*len(neighbours)
    return P


def log_density_laplacian(e, sigma, b_eta):
    """log p_t(e) of eq. (bank.weights), from eq. (NL.density): log phi(u) + log(R(k-u) + R(k+u)) - log 2b."""
    u = e/sigma                                        # u = e / sigma
    k_t = sigma/b_eta                                  # k = sigma / b_eta
    return (-0.5*u**2 - 0.5*LOG_2PI                    # log phi(u)
            + np.logaddexp(log_mills(k_t - u), log_mills(k_t + u))   # log(R(k - u) + R(k + u))
            - np.log(2*b_eta))


def bank_step(W, V, log_mu, x_t, d_t, power, grid, log_P, chi_fn, log_density, scale):
    """One step of Algorithm 1 (V given) or Algorithm 2 (V = None) for B banks side by side.
    W (K, B, L) locations, V (K, B) variances, log_mu (K, B) log weights, x_t (B, L), d_t and power (B,).
    Returns the new W, V, log_mu, and log c, log p and u = e/sigma of this step (K, B)."""
    L = W.shape[-1]
    # mixing, eqs. (bank.mixing.prob) and (bank.mixing)
    log_joint = log_P[:, :, None] + log_mu[:, None, :]   # log p_ij mu_{t-1,i}, (i, j, B)
    log_c = logsumexp(log_joint, axis=0)                 # log c_{t,j}
    mix = np.exp(log_joint - log_c[None])                # mu_{t-1,i|j}
    W_bar = np.einsum("ijb,ibm->jbm", mix, W)            # w_bar_{t,j}
    if V is not None:
        diff = W[:, None] - W_bar[None]                  # w_{t-1,i} - w_bar_{t,j}, (i, j, B, L)
        spread = np.einsum("ijbm,ijbm->ijb", diff, diff)/L
        V_bar = np.einsum("ijb,ijb->jb", mix, V[:, None] + spread)   # v_bar_{t,j}
        v_tilde = V_bar + grid[:, None]                  # v_tilde = v_bar + eps^(j)
    else:
        v_tilde = np.broadcast_to(grid[:, None], log_mu.shape)       # fKF: the fixed v^(j)
    # one sKF / fKF step per filter, eq. (bank.update)
    sigma = np.sqrt(v_tilde*power)                       # sigma_{t,j}
    e = d_t - np.einsum("jbm,bm->jb", W_bar, x_t)        # e_{t,j}, each filter's own error
    chi, slope = chi_fn(e, sigma, scale)
    W = W_bar + x_t*(chi/power)[..., None]
    if V is not None:
        V = v_tilde*(1 - slope/L)
    # weights, eq. (bank.weights), in logs
    log_p = log_density(e, sigma, scale)
    ell = log_c + log_p
    log_mu = ell - logsumexp(ell, axis=0)
    return W, V, log_mu, log_c, log_p, e/sigma


def bank_batch(X, D, h, grid, q, family, chi_fn, log_density, scale, v0=VAR_THETA_0, flip_at=None,
               wait=None, record=False, clamp=None, excess=False):
    """B banks side by side, one per row of X and D. Returns the misalignment of the output (B, n),
    the weights averaged over the rows (K, n) and the largest |u| = |e|/sigma met. record=True adds,
    per filter and step, log mu, log c, log p and the misalignment of each filter (K, B, n), and the
    output after each step (B, n, L). excess=True adds s_t = x_t'(theta_t - w_hat_{t-1}) (B, n)."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    grid = np.asarray(grid, dtype=float)
    K = len(grid)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))          # log 0 = -inf where p_ij = 0
    W = np.zeros((K, B, L))
    V = np.full((K, B), float(v0)) if family == "sKF" else None
    log_mu = np.full((K, B), -np.log(K))                 # mu_0 = 1/K
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    mu_mean = np.empty((K, n))
    u_max = 0.0
    if record:
        rec = {key: np.full((K, B, n), np.nan) for key in ("log_mu", "log_c", "log_p", "mis_j")}
        rec["w_hat"] = np.full((B, n, L), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        target = h if flip_at is None or t < flip_at else -h
        dw = w_hat - target
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_hat_{t-1})
        if record:
            dW = W - target
            rec["mis_j"][:, :, t] = np.einsum("jbm,jbm->jb", dW, dW)
        if t >= first:
            power = np.einsum("bm,bm->b", x_t, x_t)
            W, V, log_mu, log_c, log_p, u = bank_step(W, V, log_mu, x_t, D[:, t], power, grid, log_P,
                                                      chi_fn, log_density, scale)
            if clamp is not None:                        # Leszek's mu = max(mu, clamp), renormalized
                log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
                log_mu = log_mu - logsumexp(log_mu, axis=0)
            w_hat = np.einsum("jb,jbm->bm", np.exp(log_mu), W)          # eq. (bank.output)
            u_max = max(u_max, float(np.abs(u).max()))
            if record:
                rec["log_c"][:, :, t], rec["log_p"][:, :, t], rec["w_hat"][:, t] = log_c, log_p, w_hat
        mu_mean[:, t] = np.exp(log_mu).mean(axis=1)
        if record:
            rec["log_mu"][:, :, t] = log_mu
    out = {"mis": mis, "mu": mu_mean, "u_max": u_max}
    if excess:
        out["s"] = s
    if record:
        out.update(rec)
    return out


EPS_BANK = np.geomspace(1e-9, 1e-4, 6)     # the draft's remark, line 1614
V_BANK = np.geomspace(1e-6, 1e-2, 9)       # Leszek's fkfbank*.py
Q_BANK = 1e-4
# === end of the copied block ===

# Copied from bank-skf-fkf.ipynb section 4 (commit 57bd568), NOT edited.
def log_density_gaussian(e, sigma, v_eta):
    """log p_t(e) for Gaussian noise: N(0, sigma^2 + v_eta), eq. (bank.weights)."""
    s2 = sigma**2 + v_eta
    return -0.5*(LOG_2PI + np.log(s2)) - e**2/(2*s2)


print("filters ready")

In [ ]:
# === copied from kf-bank-q-sweep.ipynb section 3.1 (commit 56108a2), NOT edited ===
# === new: the bank of KFs, Algorithm 3 (kf-bank-algorithm.tex) ===
# The O(K M^2) work (the mixing of the covariances, the prediction, Sigma~ x and the covariance update) runs in two
# numba kernels, parallel over the K x B filters; chi, chi' and the log density are the scipy functions above, called
# between the two kernels, so the Laplacian pieces are the same code as in every other filter here.
# Only the upper triangle (n >= m) of every covariance is computed or read. Sigma is symmetric, so the lower triangle
# is never needed, and the symmetrisation that KF_batch does after each update is implicit.
@nb.njit(parallel=True)
def _kf_mix_predict(W, S, mix, W_bar, grid, x, S_t, Sx, s2):
    """Eq. (KF.bank.mixing) for the covariances and the first line of eq. (KF.bank.update), K filters x B banks.
    W (K, B, L) and S (K, B, L, L) the filters after step t-1; mix (K, K, B) = mu_{t-1,i|j}; W_bar (K, B, L) the
    mixed locations; x (B, L). Writes S_t = Sigma~_{t,j} = Sigma_bar_{t,j} + eps^(j) I, Sx = Sigma~ x, s2 = x' Sigma~ x."""
    K, B, L = W.shape
    for bj in nb.prange(B*K):
        b, j = bj//K, bj % K                     # bank-major order: S[i, b] is reused by j - 1, j and j + 1
        T = S_t[j, b]
        for m in range(L):
            for n in range(m, L):
                T[m, n] = 0.0
        for i in range(K):
            c = mix[i, j, b]
            if c == 0.0:                         # p_ij = 0: i is not a neighbour of j
                continue
            Si = S[i, b]
            for m in range(L):
                dm = c*(W[i, b, m] - W_bar[j, b, m])
                for n in range(m, L):            # mu_{i|j} (Sigma_i + (w_i - w_bar_j)(w_i - w_bar_j)')
                    T[m, n] += c*Si[m, n] + dm*(W[i, b, n] - W_bar[j, b, n])
        for m in range(L):
            T[m, m] += grid[j]                   # + eps^(j) I
            Sx[j, b, m] = 0.0
        for m in range(L):                       # Sigma~ x from the upper triangle
            xm = x[b, m]
            acc = T[m, m]*xm
            for n in range(m + 1, L):
                acc += T[m, n]*x[b, n]
                Sx[j, b, n] += T[m, n]*xm
            Sx[j, b, m] += acc
        tot = 0.0
        for m in range(L):
            tot += Sx[j, b, m]*x[b, m]
        s2[j, b] = tot                           # sigma_{t,j}^2 = x' Sigma~ x


@nb.njit(parallel=True)
def _kf_update(S_t, Sx, coef, S):
    """Second line of eq. (KF.bank.update), upper triangle: S = Sigma~ - coef (Sigma~ x)(Sigma~ x)', coef = chi'/sigma^2."""
    K, B, L = Sx.shape
    for bj in nb.prange(B*K):
        b, j = bj//K, bj % K
        c = coef[j, b]
        T, U = S_t[j, b], S[j, b]
        for m in range(L):
            cm = c*Sx[j, b, m]
            for n in range(m, L):
                U[m, n] = T[m, n] - cm*Sx[j, b, n]


def full_cov(S):
    """The symmetric matrices that the upper triangles S (..., L, L) stand for."""
    return np.triu(S) + np.swapaxes(np.triu(S, 1), -1, -2)


def kf_bank_step(W, S, log_mu, x_t, d_t, grid, log_P, chi_fn, log_density, scale, work):
    """One step of Algorithm 3 for B banks side by side, as bank_step does for Algorithms 1 and 2.
    W (K, B, L) locations, S (K, B, L, L) covariances (upper triangles, updated in place), log_mu (K, B), x_t (B, L),
    d_t (B,), work = (S_t, Sx, s2) buffers of shapes (K, B, L, L), (K, B, L), (K, B).
    Returns the new W and log_mu, and log c, log p and u = e/sigma of this step (K, B)."""
    # mixing, eqs. (bank.mixing.prob) and (KF.bank.mixing)
    log_joint = log_P[:, :, None] + log_mu[:, None, :]   # log p_ij mu_{t-1,i}, (i, j, B)
    log_c = logsumexp(log_joint, axis=0)                 # log c_{t,j}
    mix = np.exp(log_joint - log_c[None])                # mu_{t-1,i|j}
    W_bar = np.einsum("ijb,ibm->jbm", mix, W)            # w_bar_{t,j}
    S_t, Sx, s2 = work
    _kf_mix_predict(W, S, mix, W_bar, grid, x_t, S_t, Sx, s2)   # Sigma_bar_{t,j} + eps^(j) I, Sigma~ x, sigma^2
    # one KF step per filter, eq. (KF.bank.update)
    sigma = np.sqrt(s2)                                  # sigma_{t,j}
    e = d_t - np.einsum("jbm,bm->jb", W_bar, x_t)        # e_{t,j}, each filter's own error
    chi, slope = chi_fn(e, sigma, scale)
    W = W_bar + Sx*(chi/s2)[..., None]                   # w_{t,j}
    _kf_update(S_t, Sx, slope/s2, S)                     # Sigma_{t,j}
    # weights, eq. (bank.weights) with sigma^2 = x' Sigma~ x, in logs
    log_p = log_density(e, sigma, scale)
    ell = log_c + log_p
    log_mu = ell - logsumexp(ell, axis=0)
    return W, log_mu, log_c, log_p, e/sigma


def kf_bank_batch(X, D, h, grid, q, chi_fn, log_density, scale, v0=VAR_THETA_0, flip_at=None, wait=None,
                  record=False, clamp=None, excess=False):
    """B banks of KFs side by side (Algorithm 3), one per row of X and D, with the options and the outputs of
    bank_batch. record=True also returns the covariances after the last step (K, B, L, L), upper triangles."""
    L, (B, n) = len(h), X.shape
    first = L if wait is None else wait
    grid = np.asarray(grid, dtype=float)
    K = len(grid)
    with np.errstate(divide="ignore"):
        log_P = np.log(transition_matrix(K, q))          # log 0 = -inf where p_ij = 0
    W = np.zeros((K, B, L))
    S = np.zeros((K, B, L, L))
    S[:, :, np.arange(L), np.arange(L)] = float(v0)     # Sigma_0 = v0 I
    work = (np.zeros((K, B, L, L)), np.zeros((K, B, L)), np.zeros((K, B)))
    log_mu = np.full((K, B), -np.log(K))                 # mu_0 = 1/K
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    mis = np.empty((B, n))
    if excess:
        s = np.empty((B, n))
    mu_mean = np.empty((K, n))
    u_max = 0.0
    if record:
        rec = {key: np.full((K, B, n), np.nan) for key in ("log_mu", "log_c", "log_p", "mis_j")}
        rec["w_hat"] = np.full((B, n, L), np.nan)
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        target = h if flip_at is None or t < flip_at else -h
        dw = w_hat - target
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if excess:
            s[:, t] = -np.einsum("bm,bm->b", x_t, dw)          # s_t = x_t'(theta_t - w_hat_{t-1})
        if record:
            dW = W - target
            rec["mis_j"][:, :, t] = np.einsum("jbm,jbm->jb", dW, dW)
        if t >= first:
            W, log_mu, log_c, log_p, u = kf_bank_step(W, S, log_mu, x_t, D[:, t], grid, log_P, chi_fn, log_density,
                                                      scale, work)
            if clamp is not None:                        # Leszek's mu = max(mu, clamp), renormalized
                log_mu = np.maximum(log_mu - log_mu.max(axis=0), np.log(clamp))
                log_mu = log_mu - logsumexp(log_mu, axis=0)
            w_hat = np.einsum("jb,jbm->bm", np.exp(log_mu), W)          # eq. (bank.output)
            u_max = max(u_max, float(np.abs(u).max()))
            if record:
                rec["log_c"][:, :, t], rec["log_p"][:, :, t], rec["w_hat"][:, t] = log_c, log_p, w_hat
        mu_mean[:, t] = np.exp(log_mu).mean(axis=1)
        if record:
            rec["log_mu"][:, :, t] = log_mu
    out = {"mis": mis, "mu": mu_mean, "u_max": u_max}
    if excess:
        out["s"] = s
    if record:
        out.update(rec)
        out["W"], out["S"] = W, S
    return out


def kf_bank_reference(X, D, L, grid, q, chi_fn, log_density, scale, v0=VAR_THETA_0, wait=None):
    """Algorithm 3 line by line in plain numpy, for the checks only: full covariances, the weights in the linear
    domain, loops over j and i. Returns the output after every step (B, n, L) and log mu (K, B, n)."""
    B, n = X.shape
    K = len(grid)
    first = L if wait is None else wait
    P = transition_matrix(K, q)
    W = np.zeros((K, B, L))                                          # line 1
    S = np.broadcast_to(float(v0)*np.eye(L), (K, B, L, L)).copy()
    mu = np.full((K, B), 1.0/K)
    x_t, w_hat = np.zeros((B, L)), np.zeros((B, L))
    w_hist, log_mu_hist = np.zeros((B, n, L)), np.zeros((K, B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)                                 # line 3
        if t >= first:
            c = np.einsum("ij,ib->jb", P, mu)                        # line 5: c_{t,j}
            mix = P[:, :, None]*mu[:, None, :]/c[None]               #         mu_{t-1,i|j}
            W_bar = np.einsum("ijb,ibm->jbm", mix, W)                # line 6
            S_bar = np.zeros((K, B, L, L))
            for j in range(K):                                       # line 7
                for i in range(K):
                    d = W[i] - W_bar[j]
                    S_bar[j] += mix[i, j][:, None, None]*(S[i] + d[:, :, None]*d[:, None, :])
            ell = np.empty((K, B))
            for j in range(K):
                S_tilde = S_bar[j] + grid[j]*np.eye(L)               # line 9
                sigma2 = np.einsum("bm,bmn,bn->b", x_t, S_tilde, x_t)
                e = D[:, t] - np.einsum("bm,bm->b", x_t, W_bar[j])
                chi, slope = chi_fn(e, np.sqrt(sigma2), scale)       # line 10
                g = np.einsum("bmn,bn->bm", S_tilde, x_t)/sigma2[:, None]
                W[j] = W_bar[j] + g*chi[:, None]                     # line 11
                S[j] = S_tilde - (slope*sigma2)[:, None, None]*g[:, :, None]*g[:, None, :]
                ell[j] = np.log(c[j]) + log_density(e, np.sqrt(sigma2), scale)   # line 12
            mu = np.exp(ell - ell.max(axis=0))                       # line 14
            mu = mu/mu.sum(axis=0)
            w_hat = np.einsum("jb,jbm->bm", mu, W)                   # line 15
        w_hist[:, t] = w_hat
        log_mu_hist[:, :, t] = np.log(mu)
    return w_hist, log_mu_hist

In [ ]:
# === copied from fkf.ipynb section 3 (commit 058a9f9), NOT edited ===
def chi_minorized(e, sigma, b_eta):
    """Eq. (45) of the draft (chi.minorized): the minorized correction tau e/(tau + |e|), and the
    ratio chi/e that replaces chi' in the variance update (Table 2). The fKF uses the first only."""
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta
    ratio = tau/(tau + np.abs(e))                      # chi_min / e, in (0, 1]
    return ratio*e, ratio
# === end of the copied block ===

print("our filters ready")

### 3.2 Augusto's filters

Copied verbatim from `filters.py` on `time_varying_system` at `6a27d96` (lines 46-54, 221-227 and 383-631), nothing
edited: his parameter and output tuples, `shift`, his base filters `sKF_algorithm` (Gaussian) and `sKF_L_algorithm`
(minorized Laplacian, eq. 45), and the five filters. Each runs one realisation.

`run_scalar` runs one of them over the rows of `X`, `D` and forms the misalignment and $s_t$ from its outputs as the
batched filters form them: `h[t]` is $\boldsymbol{w}_{t-1}$ (stored before the update of step $t$) and `y[t]` is
$\boldsymbol{x}_t^{\mathsf T}\boldsymbol{w}_{t-1}$, so $s_t = \boldsymbol{x}_t^{\mathsf T}\boldsymbol{\theta}_t - y_t$. `v[t]` is
the variance before step $t$, the same number in every column.

Parameters as in his notebook (cell 5 there), which passes the true noise variance to the Gaussian filters and the true
noise scale to the Laplacian ones: here $v_\eta$ and $b_\eta = \mathrm{E}\lvert\eta_t\rvert$, with $v_0 = 2$. His notebook gives
AWU-sKF v2 an `sKF_parameters` with $\varepsilon = 10^3$, which that filter never reads; here it gets the
`AWU_sKF_parameters` of AWU-sKF v1.

In [ ]:
# === AUGUSTO: copied verbatim from filters.py on branch time_varying_system, commit 6a27d96
#     (lines 46-54, 221-227 and 383-631), NOT edited ===
sKF_parameters = namedtuple('sKF_parameters', ['label', 'epsilon', 'var_eta', 'v_tilde_0'])
AWU_sKF_parameters = namedtuple('AWU_sKF_parameters', ['label', 'var_eta', 'v_tilde_0'])
sKF_int_parameters = namedtuple('sKF_int_parameters', ['label', 'epsilon', 'var_eta', 'v_tilde_0', 'dx_factor', 'min_std_deviations'])

sKF_L_parameters = namedtuple('sKF_L_parameters', ['label', 'epsilon', 'b_eta', 'v_tilde_0'])
AWU_sKF_L_parameters = namedtuple('AWU_sKF_L_parameters', ['label', 'b_eta', 'v_tilde_0'])
sKF_L_int_parameters = namedtuple('sKF_L_int_parameters', ['label', 'epsilon', 'b_eta', 'v_tilde_0', 'dx_factor', 'min_std_deviations'])

filter_output = namedtuple('filter_output', ['y', 'e', 'h', 'v'])


@njit(cache=True, nogil=True)
def shift(new_x_sample, x_window):
  L = len(x_window)
  new_x_window = np.zeros(L)
  new_x_window[0] = new_x_sample
  new_x_window[1:] = x_window[:-1]
  return np.ascontiguousarray(new_x_window)


@njit(cache=True, nogil=True)
def sKF_algorithm(N, x, d, h0, parameters):
  h = h0
  epsilon = parameters.epsilon
  var_eta = parameters.var_eta
  v_tilde_0 = parameters.v_tilde_0

  L = len(h)
  y = np.zeros((N,))
  e = np.zeros((N,))
  xtemp = np.zeros(L)
  v=v_tilde_0
  h_hist = np.zeros((N, L))
  v_hist = np.zeros((N, L))
  # d: salida del sistema con ruido
  # y: salida estimada
  for k in range(0,N):
    xtemp = shift(x[k], xtemp)
    y[k] = np.dot(h, xtemp)
    e[k] = d[k] - y[k]
    h_hist[k] = h
    v_hist[k] = v

    if k >= L:
      # predict
      v += epsilon
      # update
      norm = xtemp @ xtemp
      s = var_eta + v * norm
      h = h + xtemp * (v * e[k]/s) # not h+= because it would mutate h0
      v = v * (1 - (v * norm) / (L * s)) 

  return filter_output(y=y, e=e, h=h_hist, v=v_hist)

@njit(cache=True, nogil=True)
def AWU_sKF_algorithm(N, x, d, h0, parameters):
  # Adaptive Weights Uncertainty Scalar Kalman Filter (AWU-sKF)
  h = np.copy(h0)
  var_eta = np.copy(parameters.var_eta)
  v_tilde_0 = np.copy(parameters.v_tilde_0)

  L = len(h)
  float_L = np.float64(L)
  y = np.zeros((N,))
  e = np.zeros((N,))
  xtemp = np.zeros(L)
  v=v_tilde_0
  h_hist = np.zeros((N, L))
  v_hist = np.zeros((N, L))
  # d: salida del sistema con ruido
  # y: salida estimada
  for k in range(0,N):
    xtemp = shift(x[k], xtemp)
    y[k] = np.dot(h, xtemp)
    e[k] = d[k] - y[k]
    h_hist[k] = h
    v_hist[k] = v

    if k >= L:
      # predict
      norm = np.dot(xtemp, xtemp)
      J_hat = (var_eta + v*norm)
      v *= (1.0 + v*norm*(e[k]**2 - J_hat)/((J_hat**2)))
      # update
      s = var_eta + v * norm
      h += xtemp * (v * e[k]/s)
      v *= (1.0 - (v * norm) / (float_L * s)) 

  return filter_output(y=y, e=e, h=h_hist, v=v_hist)

@njit(cache=True, nogil=True)
def GISV_sKF_algorithm(N, x, d, h0, parameters):
  # Gradient Informed System Variance Uncertainty Scalar Kalman Filter (GISV-sKF)
  h = np.copy(h0)
  var_eta = np.copy(parameters.var_eta)
  v = np.copy(parameters.v_tilde_0)

  L = len(h)
  float_L = np.float64(L)
  y = np.zeros((N,))
  e = np.zeros((N,))
  xtemp = np.zeros(L)
  h_hist = np.zeros((N, L))
  v_hist = np.zeros((N, L))
  # d: salida del sistema con ruido
  # y: salida estimada
  for k in range(0,N):
    xtemp = shift(x[k], xtemp)
    y[k] = np.dot(h, xtemp)
    e[k] = d[k] - y[k]
    h_hist[k] = h
    v_hist[k] = v

    if k >= L:
      # predict
      norm = np.dot(xtemp, xtemp)
      # update
      s = var_eta + v * norm
      g = (v * e[k]/s)
      h += xtemp * g 
      
      epsilon = norm * (g**2) / float_L
      v *= (1.0 - (v * norm) / (float_L * s))
      v += epsilon
      
  return filter_output(y=y, e=e, h=h_hist, v=v_hist)

@njit(cache=True, nogil=True)
def AWU_sKF_L_algorithm(N, x, d, h0, parameters):
  # Adaptive Weights Uncertainty Scalar Kalman Filter (AWU-sKF)
  h = np.copy(h0)
  b_eta = np.copy(parameters.b_eta)
  v_tilde_0 = np.copy(parameters.v_tilde_0)

  L = len(h)
  float_L = np.float64(L)
  
  y = np.zeros((N,))
  e = np.zeros((N,))
  xtemp = np.zeros(L)
  v=v_tilde_0
  h_hist = np.zeros((N, L))
  v_hist = np.zeros((N, L))
  
  for k in range(0,N):
    xtemp = shift(x[k], xtemp)
    y[k] = np.dot(h, xtemp)
    e[k] = d[k] - y[k]
    h_hist[k] = h
    v_hist[k] = v

    if k >= L:
      # predict
      norm = np.dot(xtemp, xtemp)
      var_eta = b_eta * np.abs(e[k])
      J_hat = (var_eta + v*norm)
      v *= (1.0 + v*norm*(e[k]**2 - J_hat)/((J_hat**2)))
      # update
      s = var_eta + v * norm
      h += xtemp * (v * e[k]/s)
      v *= (1.0 - (v * norm) / (float_L * s)) 

  return filter_output(y=y, e=e, h=h_hist, v=v_hist)

@njit(cache=True, nogil=True)
def AWU_sKF_algorithm_v2(N, x, d, h0, parameters):
  # Adaptive Weights Uncertainty Scalar Kalman Filter (AWU-sKF)
  h = np.copy(h0)
  v = np.copy(parameters.v_tilde_0)
  var_eta = np.copy(parameters.var_eta)

  L = len(h)
  float_L = np.float64(L)
  
  y = np.zeros((N,))
  e = np.zeros((N,))
  xtemp = np.zeros(L)
  h_hist = np.zeros((N, L))
  v_hist = np.zeros((N, L))
  
  for k in range(0,N):
    xtemp = shift(x[k], xtemp)
    y[k] = np.dot(h, xtemp)
    e[k] = d[k] - y[k]
    h_hist[k] = h
    v_hist[k] = v

    if k >= L:
      # predict
      norm = np.dot(xtemp, xtemp)
      var_e = (var_eta + v*norm)
      v *= 1.0 + v * norm * (e[k]**2)/(float_L*(var_e**2))
      # update
      s = var_eta + v * norm
      h += xtemp * (v * e[k]/s)
      v *= (1.0 - (v * norm) / (float_L * s)) 

  return filter_output(y=y, e=e, h=h_hist, v=v_hist)

@njit(cache=True, nogil=True)
def AWU_sKF_L_algorithm_v2(N, x, d, h0, parameters):
  # Adaptive Weights Uncertainty Scalar Kalman Filter (AWU-sKF)
  h = np.copy(h0)
  b_eta = np.copy(parameters.b_eta)
  v_tilde_0 = np.copy(parameters.v_tilde_0)

  L = len(h)
  float_L = np.float64(L)
  
  y = np.zeros((N,))
  e = np.zeros((N,))
  xtemp = np.zeros(L)
  v=v_tilde_0
  h_hist = np.zeros((N, L))
  v_hist = np.zeros((N, L))
  
  for k in range(0,N):
    xtemp = shift(x[k], xtemp)
    y[k] = np.dot(h, xtemp)
    e[k] = d[k] - y[k]
    h_hist[k] = h
    v_hist[k] = v

    if k >= L:
      # predict
      norm = np.dot(xtemp, xtemp)
      var_eta = b_eta * np.abs(e[k])
      var_e = (var_eta + v*norm)
      v *= 1.0 + v * norm * (e[k]**2)/(float_L*(var_e**2))
      # update
      s = var_eta + v * norm
      h += xtemp * (v * e[k]/s)
      v *= (1.0 - (v * norm) / (float_L * s)) 
      
  return filter_output(y=y, e=e, h=h_hist, v=v_hist)

@njit(cache=True, nogil=True)
def sKF_L_algorithm(N, x, d, h0, parameters):
    h = h0
    epsilon = parameters.epsilon
    b_eta = parameters.b_eta
    v_tilde_0 = parameters.v_tilde_0  

    L = len(h)
    y = np.zeros((N,))
    e = np.zeros((N,))
    xtemp = np.zeros(L)
    v=v_tilde_0
    h_hist = np.zeros((N, L))
    v_hist = np.zeros((N, L))
    # d: salida del sistema con ruido
    # y: salida estimada
    for k in range(0,N):
        xtemp = shift(x[k], xtemp)
        y[k] = np.dot(h, xtemp)
        e[k] = d[k] - y[k]
        h_hist[k] = h
        v_hist[k] = v

        if k >= L:
            # predict
            v += epsilon
            # update
            norm = np.dot(xtemp, xtemp)
            s = b_eta * abs(e[k]) + v * norm
            h = h + xtemp * (v * e[k]/s) # not h+= because it would mutate h0
            v = v * (1 - (v * norm) / (L * s)) 

    return filter_output(y=y, e=e, h=h_hist, v=v_hist)
# === end of the copied block ===


def run_scalar(fn, params, X, D, flip_at=None):
    """One of Augusto's filters over the rows of X and D, one realisation at a time (his code is scalar). Returns the
    misalignment and s_t (B, n), formed from his outputs as the batched filters form them, and his v before each step."""
    B, n = X.shape
    clean = clean_output(X, change_at=flip_at)                   # x_t' theta_t
    cut = n if flip_at is None else flip_at
    mis, s, v = np.empty((B, n)), np.empty((B, n)), np.empty((B, n))
    for b in range(B):
        out = fn(n, np.ascontiguousarray(X[b]), np.ascontiguousarray(D[b]), np.zeros(M), params)
        dw = out.h                                               # h[t] = w_{t-1}, stored before the update of step t
        dw[:cut] -= ho                                           # w_{t-1} - theta_t: theta = h_o, and -h_o from flip_at on
        dw[cut:] += ho
        mis[b] = np.einsum("tm,tm->t", dw, dw)
        s[b] = clean[b] - out.y                                  # s_t = x_t' theta_t - x_t' w_{t-1}
        v[b] = out.v[:, 0]                                       # the same v in every column
    return mis, s, v


AUGUSTO = {"AWU-sKF v1": (AWU_sKF_algorithm, "Gaussian"), "AWU-sKF v2": (AWU_sKF_algorithm_v2, "Gaussian"),
           "GISV-sKF": (GISV_sKF_algorithm, "Gaussian"), "AWU-sKF-L v1": (AWU_sKF_L_algorithm, "Laplacian"),
           "AWU-sKF-L v2": (AWU_sKF_L_algorithm_v2, "Laplacian")}
BASE = {"sKF, Gaussian": sKF_algorithm, "sKF-L, minorized": sKF_L_algorithm}      # his filters with a fixed eps


def augusto_params(name, v_eta, b_eta):
    # GISV-sKF reads var_eta and v_tilde_0 only, as AWU-sKF v1 and v2
    if AUGUSTO[name][1] == "Gaussian":
        return AWU_sKF_parameters(label=name, var_eta=v_eta, v_tilde_0=VAR_THETA_0)
    return AWU_sKF_L_parameters(label=name, b_eta=b_eta, v_tilde_0=VAR_THETA_0)


def base_params(name, eps, v_eta, b_eta):
    if name == "sKF, Gaussian":
        return sKF_parameters(label=name, epsilon=float(eps), var_eta=v_eta, v_tilde_0=VAR_THETA_0)
    return sKF_L_parameters(label=name, epsilon=float(eps), b_eta=b_eta, v_tilde_0=VAR_THETA_0)


print("Augusto's filters ready")

## 4. Checks

### 4.1 His base filters are `sKF_batch` with the Gaussian and the minorized $\chi$

`sKF_algorithm` must be `sKF_batch` with `chi_gaussian` ($v_\eta$), `sKF_L_algorithm` must be `sKF_batch` with
`chi_minorized` ($b_\eta$), and `run_scalar` must form the misalignment and $s_t$ as `sKF_batch` does. Then the search of
section 5, run with `sKF_batch`, tunes his base filters, and his filters are measured exactly like ours. Realisations 0-2
of AR($-0.9$), $\beta^* = 0.2$, 3000 steps, the flip at 1500, three values of $\varepsilon$.

In [ ]:
X_c41, D_c41 = stack([signals(seed, -0.9, 3000, change_at=1500) for seed in range(3)])
v_eta_c41, b_eta_c41 = noise_variance(-0.9), noise_of(-0.9)[1]
print(f"{'filter':<18}{'eps':>8}{'misalignment':>15}{'s_t':>11}")
for name, chi_fn, scale in (("sKF, Gaussian", chi_gaussian, v_eta_c41), ("sKF-L, minorized", chi_minorized, b_eta_c41)):
    for eps in (1e-7, 1e-6, 1e-5):
        mis_a, s_a, _ = run_scalar(BASE[name], base_params(name, eps, v_eta_c41, b_eta_c41), X_c41, D_c41, flip_at=1500)
        mis_b, s_b = sKF_batch(X_c41, D_c41, ho, scale, np.full(3, eps), chi_fn, flip_at=1500, excess=True)
        print(f"{name:<18}{eps:>8.0e}{np.max(np.abs(mis_a - mis_b)/mis_b):>15.1e}"
              f"{np.max(np.abs(s_a - s_b))/np.max(np.abs(s_b)):>11.1e}")
print("   misalignment: largest relative difference; s_t: largest |difference| / largest |s_t|")

### 4.2 GISV-sKF adds $\lVert\boldsymbol{w}_t - \boldsymbol{w}_{t-1}\rVert^2/M$

From its own outputs: the $v$ it stores before step $t+1$ must be the sKF update of step $t$ plus
$\hat\varepsilon_t = \lVert\boldsymbol{w}_t - \boldsymbol{w}_{t-1}\rVert^2/M$, the estimator of the draft's Remark 5.
Realisation 0 of the check above. Printed beside it: the median of $\hat\varepsilon_t/v_t$ over the last 1000 steps.

In [ ]:
x0, d0 = X_c41[0], D_c41[0]
n0 = len(x0)
out = GISV_sKF_algorithm(n0, x0, d0, np.zeros(M), augusto_params("GISV-sKF", v_eta_c41, b_eta_c41))
power = np.convolve(x0**2, np.ones(M))[:n0]                         # ||x_t||^2 of the windows
v_before = out.v[:-1, 0]                                             # v before step t
step = out.h[1:] - out.h[:-1]                                        # w_t - w_{t-1}
eps_hat = np.einsum("tm,tm->t", step, step)/M
v_next = v_before*(1 - v_before*power[:-1]/(M*(v_eta_c41 + v_before*power[:-1]))) + eps_hat
t_up = np.arange(M, n0 - 1)                                          # the steps that update
v_after = out.v[1:, 0]
print(f"GISV-sKF: v before step t+1 against the sKF update of step t + ||w_t - w_(t-1)||^2/M, largest relative "
      f"difference {np.max(np.abs(v_after[t_up] - v_next[t_up])/v_after[t_up]):.1e}")
print(f"   median of eps_hat_t / v_t over the last 1000 steps: {np.median(eps_hat[-1000:]/v_after[-1000:]):.3f}")
del X_c41, D_c41, out, step

### 4.3 His five filters in his own setup

In our scenario the AWU-sKF-L filters do not follow the flip (sections 6-11). To see whether that comes from our
scenario: his five filters in his own setup, cell 2 of `time_varying_systems.ipynb` at `6a27d96`. $L = 16$, the
normalised sinc response, white input of unit variance, $10^5$ steps with $\boldsymbol{h} \to -\boldsymbol{h}$ in the middle,
$v_0 = 1$, and his two noises, Gaussian of variance $10^{-3}$ and Laplacian of scale $\sqrt{10^{-3}}$ (variance
$2 \times 10^{-3}$). His parameters: $v_\eta$ = the noise variance, $b_\eta = \sqrt{10^{-3}}$. Eight realisations, drawn with
numpy (his own noise generator is not needed for these two densities). A check, not a comparison: none of our filters
runs here.

In [ ]:
def augusto_setup(noise, seed, n=100_000, L=16):
    # his sys_inversion environments: white input of unit variance, the normalised sinc, h -> -h at n//2
    h16 = np.sinc(np.linspace(0, 1, L))
    h16 = h16/np.linalg.norm(h16)
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n)
    y = np.convolve(h16, x)[:n]
    y[n//2:] = -y[n//2:]
    eta = np.sqrt(1e-3)*rng.standard_normal(n) if noise == "Gaussian" else rng.laplace(0.0, np.sqrt(1e-3), n)
    return h16, x, y + eta


start = time.time()
n16, H16 = 100_000, 50_000
print(f"{'noise':<11}{'filter':<14}{'before':>8}{'+1000':>8}{'+5000':>8}{'+20000':>8}{'end':>8}{'mean':>8}"
      f"{'v flip':>10}{'v end':>10}")
for noise, v_eta16 in (("Gaussian", 1e-3), ("Laplacian", 2e-3)):
    for name, (fn, likelihood) in AUGUSTO.items():
        params = (AWU_sKF_parameters(label=name, var_eta=v_eta16, v_tilde_0=1.0) if likelihood == "Gaussian"
                  else AWU_sKF_L_parameters(label=name, b_eta=np.sqrt(1e-3), v_tilde_0=1.0))
        mis, v_flip, v_end = [], [], []
        for seed in range(8):
            h16, x, d = augusto_setup(noise, 100 + seed)
            out = fn(n16, x, d, np.zeros(16), params)
            theta = np.where(np.arange(n16)[:, None] < H16, h16, -h16)
            mis.append(((out.h - theta)**2).sum(axis=1))
            v_flip.append(out.v[H16, 0])
            v_end.append(out.v[-1, 0])
        m = np.mean(mis, axis=0)
        levels = (m[3*H16//4:H16].mean(), m[H16 + 1000], m[H16 + 5000], m[H16 + 20000], m[-H16//4:].mean(), m.mean())
        print(f"{noise:<11}{name:<14}" + "".join(f"{10*np.log10(v):>8.1f}" for v in levels)
              + f"{np.median(v_flip):>10.1e}{np.median(v_end):>10.1e}")
print("   mean misalignment over the 8 realisations [dB]: before = the last quarter before the flip; then 1000, 5000 and "
      "20000 steps after it;\n   end = the last quarter; mean = over the record. v: median over the realisations, "
      "before the flip and at the end")
del out, theta, mis
print(f"[{time.time() - start:.0f} s]")

## 5. Measurement

Sections 5 and 5.1 of `kf-bank-q-sweep.ipynb` (commit `56108a2`). One option added to `cached`, `folder` (default
`results/kf-bank-q-sweep/`), so that the runs of ours are read from where that notebook saved them; nothing else
edited.

New below them:
- `BEST_Q`: each bank's best $q$ per scenario, as the findings of `kf-bank-q-sweep.ipynb` mark it (the lowest mean over
  the record, from the mean curves), checked against the saved sweep. `PRINTED_MEAN`: the means over the record printed
  there, the control.
- `base_values`: notebook 09's search for the two base filters (`sweep` and `pick`, as `tuned_values` does for ours),
  with `sKF_batch` and `chi_gaussian` or `chi_minorized`; check 4.1 shows that these are his base filters.
- `scalar_run`: one of his filters, or a base filter, rest run and change run through `run_scalar`, summarised by
  `summarise` (the same numbers and curves as ours), plus the quartiles of $v_t$ over the change run. Saved under
  `results/awu-gisv-vs-banks/`.
- `scenario_compare`: all the filters of one scenario, the control and two tables, from the mean and from the median
  curves.

In [ ]:
# === copied from kf-bank-q-sweep.ipynb section 5 (commit 56108a2), NOT edited ===
# === copied from bank-kf-excess-error.ipynb section 5 (commit f2ded06), NOT edited ===
# === IGNACIO: steady_state, is_settled and recovery - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized, notebooks/09_matcheado.ipynb, commit 6e6dabc,
#         via bank-skf-fkf.ipynb (commit 57bd568)
def steady_state(misalignment):
    """Floor in dB, and the first step within 3 dB of it."""
    tail = slice(3*len(misalignment)//4, len(misalignment))
    floor = 10*np.log10(misalignment[tail].mean())
    db = 10*np.log10(misalignment)
    reached = int(np.argmax(db < floor + 3))
    return floor, reached


def is_settled(misalignment, drift):
    """Settled within the run: within 3 dB of the floor by half of it, and no longer descending."""
    n = len(misalignment)
    floor, reached = steady_state(misalignment)
    third_quarter = 10*np.log10(misalignment[n//2:3*n//4].mean())
    # reached = 0: the run never left the 3 dB band around its start, so it never converged either
    return bool(0 < reached <= n/2 and third_quarter - floor <= drift)


def recovery(misalignment, floor):
    """Steps from the change until the misalignment is back within 3 dB of the floor at rest;
    -1 if it never gets there before the run ends."""
    db = 10*np.log10(misalignment[N:])
    back = db < floor + 3
    if not back.any():
        return -1
    return int(np.argmax(back))
# === end of the copied block ===


# === copied from bank-skf-fkf.ipynb sections 7 and 7.2 (commit 57bd568), NOT edited ===
def time_to(misalignment, level_db):
    """First step after the change below level_db; -1 if never."""
    below = 10*np.log10(misalignment[N:]) < level_db
    return int(np.argmax(below)) if below.any() else -1


LEVELS = [0.0, -5.0, -10.0, -15.0]


def first_below(misalignment, level_db):
    below = 10*np.log10(misalignment) < level_db
    return int(np.argmax(below)) if below.any() else -1


def measure(rest_curve, change_curve):
    floor, reached = steady_state(rest_curve)
    return dict(floor=floor, reached=reached, settled=is_settled(rest_curve, DRIFT_DB),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS], rec=recovery(change_curve, floor),
                back_20=time_to(change_curve, TARGET_DB), mean_db=10*np.log10(change_curve.mean()),
                curve=change_curve)


def show_table(rows):
    print(f"{'':<34}{'at rest':-^46}  {'after the flip':-^50}")
    print(f"{'filter':<22}{'eps or v':>12}{'floor':>8}{'settled':>9}{'own+3':>7}{'-17 dB':>8}{'-20 dB':>8}"
          + "".join(f"{level:>8.0f}" for level in LEVELS) + f"{'own+3':>9}{'-20 dB':>9}{'mean [dB]':>11}")
    for name, r in rows.items():
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{'yes' if r['settled'] else 'NO':>9}{r['reached']:>7d}"
              f"{r['to_17']:>8d}{r['to_20']:>8d}" + "".join(f"{s:>8d}" for s in r["levels"])
              + f"{r['rec']:>9d}{r['back_20']:>9d}{r['mean_db']:>11.2f}")
    print("   steps; -1 = never within the run")


EPS_GRID = np.logspace(-8, -3, 11)          # notebook 09


def pick(grid, floors, settled, target):
    values, kept = grid[settled], floors[settled]
    if target < kept.min():
        return values[np.argmin(kept)], "target below every settled floor"
    if target > kept.max():
        return values[np.argmax(kept)], "target above every settled floor"
    order = np.argsort(kept)
    return 10**np.interp(target, kept[order], np.log10(values)[order]), "ok"
# === end of the copied block ===


def sweep_kf(b_eta, X, D):
    # notebook 09's search for the KF, as sweep() of the bank notebook, one grid value at a time
    floors, settled = [], []
    for eps in EPS_GRID:
        curve = KF_batch(X, D, ho, b_eta, np.full(len(X), eps))[0].mean(axis=0)
        floors.append(steady_state(curve)[0])
        settled.append(is_settled(curve, DRIFT_DB))
    return np.array(floors), np.array(settled)


SMOOTH = 101                # centred moving average on both means, in steps


def smoothed_ratio(num, den, cut=None):
    """Moving average of both means, then the ratio. No window crosses step cut (the change)."""
    if cut is not None:
        return np.concatenate([smoothed_ratio(num[:cut], den[:cut]), smoothed_ratio(num[cut:], den[cut:])])
    return uniform_filter1d(num, SMOOTH, mode="nearest")/uniform_filter1d(den, SMOOTH, mode="nearest")


def measure_excess(s_rest, s_change, den_rest, den_change):
    """s_rest (R, N), s_change (R, 2N); den_* = E[(x_t' theta_t)^2], the mean over realisations at each step."""
    num_rest, num_change = (s_rest**2).mean(axis=0), (s_change**2).mean(axis=0)
    rest_curve, change_curve = smoothed_ratio(num_rest, den_rest), smoothed_ratio(num_change, den_change, cut=N)
    tail = slice(3*N//4, N)                            # the window of steady_state
    return dict(floor=10*np.log10(num_rest[tail].mean()/den_rest[tail].mean()),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS + [TARGET_DB]],
                mean_db=10*np.log10(num_change.mean()/den_change.mean()),
                rest_curve=rest_curve, curve=change_curve)


def show_excess(rows):
    print(f"{'':<34}{'at rest':-^37}  {'after the flip, excess error':-^40}")
    print(f"{'filter':<22}{'eps or v':>12}{'mis.':>8}{'excess':>8}{'gap':>7}{'-17 dB':>7}{'-20 dB':>7}"
          + "".join(f"{level:>8.0f}" for level in LEVELS + [TARGET_DB]) + f"{'mean mis.':>11}{'mean exc.':>11}")
    for name, r in rows.items():
        x = r["excess"]
        value = "grid" if np.isnan(r["value"]) else f"{r['value']:.4e}"
        print(f"{name:<22}{value:>12}{r['floor']:>8.2f}{x['floor']:>8.2f}{x['floor'] - r['floor']:>7.2f}"
              f"{x['to_17']:>7d}{x['to_20']:>7d}" + "".join(f"{s:>8d}" for s in x["levels"])
              + f"{r['mean_db']:>11.2f}{x['mean_db']:>11.2f}")
    print("   floors and means in dB; mis. = misalignment, excess = excess error, gap = excess - mis.")
    print("   steps on the excess error (smoothed, good to about 50 steps); -1 = never within the run")


def run_filter(name, value, b_eta, X, D, flip_at=None):
    # misalignment and s_t, (B, n) each
    family, method = name.split(", ")
    if method == "bank":
        out = bank_batch(X, D, ho, EPS_BANK if family == "sKF" else V_BANK, Q_BANK, family, chi_laplacian,
                         log_density_laplacian, b_eta, flip_at=flip_at, excess=True)
        return out["mis"], out["s"]
    values = np.full(len(X), value)
    if family == "sKF":
        return sKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    if family == "fKF":
        return fKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    mis, _, s = KF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    return mis, s


# floor, own + 3, recovery and mean over the record printed by bank-skf-fkf.ipynb (commit 57bd568)
BANK_NB = {(-0.9, "sKF, exact"): (-20.01, 4756, 7584, -14.08), (-0.9, "sKF, bank"): (-25.92, 68043, 75315, -13.89),
           (-0.9, "fKF, exact"): (-19.99, 5461, 8188, -13.45), (-0.9, "fKF, bank"): (-15.68, 28234, 29311, -11.92),
           (0.0, "sKF, exact"): (-20.01, 1564, 1221, -17.22), (0.0, "sKF, bank"): (-44.92, 6963, 6433, -22.64),
           (0.0, "fKF, exact"): (-20.00, 1056, 1596, -16.67), (0.0, "fKF, bank"): (-44.58, 12135, 12233, -23.01)}


def scenario_run(ar_a, tuned):
    """The KF tuned to -20 dB, then the five filters at rest and across the change, with s_t."""
    start = time.time()
    (X_r, D_r), (X_c, D_c) = scenario_signals(ar_a)
    b_eta = noise_of(ar_a)[1]
    den_r = (clean_output(X_r)**2).mean(axis=0)
    den_c = (clean_output(X_c, change_at=N)**2).mean(axis=0)
    print(f"mean of (x_t' theta_t)^2 over the change runs: {den_c[M:].mean():.4f}; "
          f"h' R_xx h = {signal_power(ar_a):.4f}")
    floors, settled = sweep_kf(b_eta, X_r, D_r)
    eps_kf, status = pick(EPS_GRID, floors, settled, TARGET_DB)
    print("KF: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in zip(floors, settled))
          + "   (* not settled)")
    print(f"   at -20 dB: eps = {eps_kf:.4e}, {status}   [{time.time() - start:.0f} s]")
    values = {"sKF, exact": tuned["sKF"], "sKF, bank": np.nan, "fKF, exact": tuned["fKF"], "fKF, bank": np.nan,
              "KF, exact": eps_kf}
    rows = {}
    for name, value in values.items():
        mis_r, s_r = run_filter(name, value, b_eta, X_r, D_r)
        mis_c, s_c = run_filter(name, value, b_eta, X_c, D_c, flip_at=N)
        rows[name] = dict(value=value, rest_curve=mis_r.mean(axis=0), **measure(mis_r.mean(axis=0), mis_c.mean(axis=0)),
                          excess=measure_excess(s_r, s_c, den_r, den_c))
        print(f"{name} done [{time.time() - start:.0f} s]")
    print("\ncontrol: the four filters of the bank notebook against its printed numbers (ours / printed)")
    for name, r in rows.items():
        if (ar_a, name) in BANK_NB:
            floor, steps, rec, mean = BANK_NB[(ar_a, name)]
            print(f"   {name:<11} floor {r['floor']:.2f} / {floor:.2f}, own+3 {r['reached']} / {steps}, "
                  f"recovery {r['rec']} / {rec}, mean {r['mean_db']:.2f} / {mean:.2f}")
    print("\nmisalignment")
    show_table(rows)
    print("\nexcess error")
    show_excess(rows)
    return rows


print("measurement ready")

In [ ]:
# === copied from kf-bank-q-sweep.ipynb section 5.1 (commit 56108a2); one option added to cached (folder,
#     default CACHE), nothing else edited ===
Q_SWEEP = [1e-4, 1e-3, 1e-2, 1e-1, 1/3]       # 1e-4 is the value of every earlier bank run
V_GRID = np.logspace(-6, -1, 11)               # fkf.ipynb, leszek-setup.ipynb; the fKF search of bank-skf-fkf.ipynb


# === copied from bank-skf-fkf.ipynb section 7.2 (commit 57bd568), NOT edited ===
def sweep(family, chi_fn, scale, grid, X, D):
    # notebook 09's search, as clipped-matched.ipynb section 5: every grid point at R realisations at once
    g = len(grid)
    Xg, Dg, values = np.repeat(X, g, axis=0), np.repeat(D, g, axis=0), np.tile(grid, len(X))
    mis = (sKF_batch(Xg, Dg, ho, scale, values, chi_fn) if family == "sKF"
           else fKF_batch(Xg, Dg, ho, scale, values, chi_fn))
    curves = mis.reshape(len(X), g, -1).mean(axis=0)
    del mis
    return (np.array([steady_state(c)[0] for c in curves]),
            np.array([is_settled(c, DRIFT_DB) for c in curves]))
# === end of the copied block ===


# The single filters at beta* = 0.2, tuned to -20 dB in the earlier notebooks and given here, not re-tuned: sKF and fKF
# from bank-skf-fkf.ipynb, the KF from bank-kf-excess-error.ipynb (printed there to five digits).
TUNED_HEAVY = {-0.9: {"sKF": 1.3493e-6, "fKF": 2.985e-4, "KF": 8.8577e-7},
               0.0: {"sKF": 1.319e-6, "fKF": 3.017e-4, "KF": 1.6553e-6}}
# floor, own+3, recovery and mean over the record printed by bank-kf-excess-error.ipynb (commit f2ded06)
PREVIOUS = dict(BANK_NB)
PREVIOUS.update({(-0.9, "KF, exact"): (-20.01, 1254, 1972, -15.63), (0.0, "KF, exact"): (-20.02, 603, 791, -17.65)})

CACHE = Path("results/kf-bank-q-sweep")        # one file per run, written as it finishes; delete it to rerun all
WINDOW = slice(N - 2000, N + 20000)            # the change run kept at full resolution: -0.25 s to +2.5 s
THIN = 4                                       # the rest run kept every THIN steps


def cached(key, compute, folder=CACHE):
    path = folder/f"{key}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    result = compute()
    folder.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    return result


def tuned_values(ar_a, beta, b_eta, X, D):
    """The single filters' -20 dB values: given at beta* = 0.2, from notebook 09's search at any other beta*
    (the searches of the bank notebook: settled points only, interpolated)."""
    if beta == BETA:
        return dict(TUNED_HEAVY[ar_a])

    def search():
        found = {}
        for family, grid in (("sKF", EPS_GRID), ("fKF", V_GRID), ("KF", EPS_GRID)):
            floors, settled = (sweep_kf(b_eta, X, D) if family == "KF"
                               else sweep(family, chi_laplacian, b_eta, grid, X, D))
            if settled.any():
                value, status = pick(grid, floors, settled, TARGET_DB)
            else:                                      # pick needs one settled point; say so instead of failing
                value, status = pick(grid, floors, np.ones(len(grid), dtype=bool), TARGET_DB)
                status = "NO grid point settled; picked among all of them, " + status
            found[family] = dict(floors=floors, settled=settled, value=value, status=status)
        return found
    found = cached(f"search_ar{ar_a:g}_beta{beta:g}", search)
    for family, r in found.items():
        print(f"{family}: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in
                                                     zip(r["floors"], r["settled"])) + "   (* not settled)")
        print(f"   at -20 dB: {r['value']:.4e}, {r['status']}")
    return {family: r["value"] for family, r in found.items()}


def run_one(name, value, q, b_eta, X, D, flip_at=None):
    """Misalignment and s_t (B, n) of one filter, and its weights averaged over the rows (K, n) if it is a bank.
    run_filter of bank-kf-excess-error.ipynb, with the banks' q as a parameter and the bank of KFs added."""
    family, method = name.split(", ")
    if method == "bank":
        out = (kf_bank_batch(X, D, ho, EPS_BANK, q, chi_laplacian, log_density_laplacian, b_eta, flip_at=flip_at,
                             excess=True) if family == "KF" else
               bank_batch(X, D, ho, EPS_BANK if family == "sKF" else V_BANK, q, family, chi_laplacian,
                          log_density_laplacian, b_eta, flip_at=flip_at, excess=True))
        return out["mis"], out["s"], out["mu"]
    values = np.full(len(X), value)
    if family == "sKF":
        mis, s = sKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    elif family == "fKF":
        mis, s = fKF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    else:
        mis, _, s = KF_batch(X, D, ho, b_eta, values, chi_laplacian, flip_at=flip_at, excess=True)
    return mis, s, None


def smooth_split(a, cut=None):
    """The 101-step centred moving average of smoothed_ratio, along the last axis; no window crosses step cut."""
    if cut is not None:
        return np.concatenate([smooth_split(a[..., :cut]), smooth_split(a[..., cut:])], axis=-1)
    return uniform_filter1d(a, SMOOTH, axis=-1, mode="nearest")


def measure_excess_median(num_rest, num_change, den_rest, den_change):
    """measure_excess for the median: num_* are the medians over the realisations of each realisation's s_t^2,
    already smoothed (the median of 20 raw squares is noisy and reads low); den_* the means of (x_t' theta_t)^2."""
    rest_curve, change_curve = num_rest/smooth_split(den_rest), num_change/smooth_split(den_change, cut=N)
    tail = slice(3*N//4, N)                            # the window of steady_state
    return dict(floor=10*np.log10(num_rest[tail].mean()/den_rest[tail].mean()),
                to_17=first_below(rest_curve, TARGET_DB + 3), to_20=first_below(rest_curve, TARGET_DB),
                levels=[time_to(change_curve, level) for level in LEVELS + [TARGET_DB]],
                mean_db=10*np.log10(num_change.mean()/den_change.mean()),
                rest_curve=rest_curve, curve=change_curve)


def summarise(mis_r, s_r, mis_c, s_c, mu_c, den_r, den_c):
    """Numbers and curves of one filter in one scenario, from its rest run (R, N) and its change run (R, 2N).
    mean: the mean over the realisations at each step, measured as in the earlier notebooks (measure,
    measure_excess). median: the median over the realisations at each step, of the misalignment and of each
    realisation's s_t^2 smoothed over 101 steps, measured by the same functions. The 25 % and 75 % curves are kept
    for the bands of the figures."""
    pct = (25, 50, 75)
    mis_q_r, mis_q_c = np.percentile(mis_r, pct, axis=0), np.percentile(mis_c, pct, axis=0)
    sm_r, sm_c = smooth_split(s_r**2), smooth_split(s_c**2, cut=N)
    exc_q_r, exc_q_c = np.percentile(sm_r, pct, axis=0), np.percentile(sm_c, pct, axis=0)
    mean_r, mean_c = mis_r.mean(axis=0), mis_c.mean(axis=0)
    out = {"mean": measure(mean_r, mean_c), "median": measure(mis_q_r[1], mis_q_c[1]),
           "mean_x": measure_excess(s_r, s_c, den_r, den_c),
           "median_x": measure_excess_median(exc_q_r[1], exc_q_c[1], den_r, den_c)}
    den_sm_r, den_sm_c = smooth_split(den_r), smooth_split(den_c, cut=N)
    f32 = lambda a: np.asarray(a, dtype=np.float32)
    out["curves"] = {
        "mis_mean": f32(mean_c[WINDOW]), "mis_q": f32(mis_q_c[:, WINDOW]),
        "exc_mean": f32(out["mean_x"]["curve"][WINDOW]), "exc_q": f32((exc_q_c/den_sm_c)[:, WINDOW]),
        "rest_mis_mean": f32(mean_r[::THIN]), "rest_mis_q": f32(mis_q_r[:, ::THIN]),
        "rest_exc_mean": f32(out["mean_x"]["rest_curve"][::THIN]), "rest_exc_q": f32((exc_q_r/den_sm_r)[:, ::THIN]),
        "mu": None if mu_c is None else f32(mu_c[:, ::8])}
    for key in ("mean", "median", "mean_x", "median_x"):         # the full-length curves are not kept
        out[key].pop("curve", None)
        out[key].pop("rest_curve", None)
    return out


def filter_run(name, value, q, ar_a, beta, sig, b_eta, den):
    """One filter in one scenario, rest and change runs, summarised; read from the cache when it has been run."""
    tag = "grid" if np.isnan(value) else f"{value:.6e}"
    key = f"{name.replace(', ', '-')}_{tag}_q{'-' if q is None else f'{q:g}'}_ar{ar_a:g}_beta{beta:g}"

    def compute():
        start = time.time()
        (X_r, D_r), (X_c, D_c) = sig
        mis_r, s_r, _ = run_one(name, value, q, b_eta, X_r, D_r)
        mis_c, s_c, mu_c = run_one(name, value, q, b_eta, X_c, D_c, flip_at=N)
        out = summarise(mis_r, s_r, mis_c, s_c, mu_c, *den)
        out.update(name=name, value=value, q=q, ar_a=ar_a, beta=beta, seconds=time.time() - start)
        return out
    return cached(key, compute)


def show_sweep(rows, stat):
    """The numbers of one scenario, from the mean curves (stat = "mean") or the median curves (stat = "median")."""
    print(f"{'':<22}{' misalignment, ' + stat + ' ':-^72}  {' excess error, ' + stat + ' ':-^38}")
    print(f"{'filter':<12}{'q':>10}{'floor':>8}{'sett.':>6}{'own+3':>7}{'-17':>7}{'0':>7}{'-10':>7}{'-15':>7}"
          f"{'rec.':>7}{'-20':>7}{'mean':>8}  {'floor':>7}{'-10':>7}{'-15':>7}{'-20':>7}{'mean':>8}")
    for (name, q), r in rows.items():
        m, x = r[stat], r[stat + "_x"]
        print(f"{name:<12}{'' if q is None else f'{q:.3g}':>10}{m['floor']:>8.2f}{'yes' if m['settled'] else 'NO':>6}"
              f"{m['reached']:>7d}{m['to_17']:>7d}{m['levels'][0]:>7d}{m['levels'][2]:>7d}{m['levels'][3]:>7d}"
              f"{m['rec']:>7d}{m['back_20']:>7d}{m['mean_db']:>8.2f}  {x['floor']:>7.2f}{x['levels'][2]:>7d}"
              f"{x['levels'][3]:>7d}{x['levels'][4]:>7d}{x['mean_db']:>8.2f}")
    print("   floors and means in dB, everything else in steps; own+3 (to the own floor + 3 dB) and -17 from zero at "
          "rest;\n   0, -10, -15, rec. (back to the own floor + 3 dB) and -20 dB after the flip; -1 = never within "
          "the run")


def scenario_sweep(ar_a, beta):
    """The three single filters at -20 dB and the three banks at each q of Q_SWEEP, in one scenario."""
    start = time.time()
    sig = scenario_signals(ar_a, beta)
    (X_r, D_r), (X_c, D_c) = sig
    b_eta = noise_of(ar_a, beta)[1]
    den = ((clean_output(X_r)**2).mean(axis=0), (clean_output(X_c, change_at=N)**2).mean(axis=0))
    print(f"b_eta = E|eta| = {b_eta:.4e}; mean of (x_t' theta_t)^2 over the change runs {den[1][M:].mean():.4f}, "
          f"h' R_xx h = {signal_power(ar_a):.4f}")
    tuned = tuned_values(ar_a, beta, b_eta, X_r, D_r)
    rows = {}
    for family in ("sKF", "fKF", "KF"):
        rows[(f"{family}, exact", None)] = filter_run(f"{family}, exact", tuned[family], None, ar_a, beta, sig, b_eta,
                                                      den)
    for q in Q_SWEEP:
        for family in ("sKF", "fKF", "KF"):
            rows[(f"{family}, bank", q)] = filter_run(f"{family}, bank", np.nan, q, ar_a, beta, sig, b_eta, den)
        print(f"q = {q:.3g} done [{time.time() - start:.0f} s]", flush=True)
    if beta == BETA:
        print("\ncontrol: the filters run before, against their printed numbers (ours / printed)")
        for (name, q), r in rows.items():
            if (ar_a, name) in PREVIOUS and q in (None, Q_BANK):
                floor, steps, rec, mean = PREVIOUS[(ar_a, name)]
                m = r["mean"]
                print(f"   {name:<11} floor {m['floor']:.2f} / {floor:.2f}, own+3 {m['reached']} / {steps}, "
                      f"recovery {m['rec']} / {rec}, mean {m['mean_db']:.2f} / {mean:.2f}")
    print()
    show_sweep(rows, "mean")
    print()
    show_sweep(rows, "median")
    print(f"[{time.time() - start:.0f} s; the 18 runs took {sum(r['seconds'] for r in rows.values())/60:.0f} min when they were run (a run read from results/ keeps its own time)]")
    return rows


results = {}
print("sweep ready")

In [ ]:
NEW_CACHE = Path("results/awu-gisv-vs-banks")    # the runs of Augusto's filters and of the base filters

# Each bank's best q: the lowest mean over the record (mean curves), as the findings of kf-bank-q-sweep.ipynb mark it
BEST_Q = {(-0.9, BETA): {"sKF": 1e-1, "fKF": 1e-4, "KF": 1e-3}, (0.0, BETA): {"sKF": 1e-4, "fKF": 1e-4, "KF": 1e-3},
          (-0.9, BETA_GAUSS): {"sKF": 1e-2, "fKF": 1/3, "KF": 1e-4},
          (0.0, BETA_GAUSS): {"sKF": 1e-4, "fKF": 1e-4, "KF": 1e-4}}

# The control: the mean over the record [dB], mean curves, printed by kf-bank-q-sweep.ipynb (its sections 6-9)
PRINTED_MEAN = {
    (-0.9, BETA): {("sKF, exact", None): -14.08, ("fKF, exact", None): -13.45, ("KF, exact", None): -15.63,
                   ("sKF, bank", 1e-4): -13.89, ("fKF, bank", 1e-4): -11.92, ("KF, bank", 1e-4): -20.55,
                   ("sKF, bank", 1e-1): -16.52, ("KF, bank", 1e-3): -20.57},
    (0.0, BETA): {("sKF, exact", None): -17.22, ("fKF, exact", None): -16.67, ("KF, exact", None): -17.65,
                  ("sKF, bank", 1e-4): -22.64, ("fKF, bank", 1e-4): -23.01, ("KF, bank", 1e-4): -23.92,
                  ("KF, bank", 1e-3): -23.94},
    (-0.9, BETA_GAUSS): {("sKF, exact", None): -6.77, ("fKF, exact", None): -8.00, ("KF, exact", None): -6.79,
                         ("sKF, bank", 1e-4): -8.68, ("fKF, bank", 1e-4): -8.00, ("KF, bank", 1e-4): -14.09,
                         ("sKF, bank", 1e-2): -10.53, ("fKF, bank", 1/3): -9.46},
    (0.0, BETA_GAUSS): {("sKF, exact", None): -11.56, ("fKF, exact", None): -10.27, ("KF, exact", None): -12.06,
                        ("sKF, bank", 1e-4): -18.55, ("fKF, bank", 1e-4): -17.09, ("KF, bank", 1e-4): -18.89}}

OURS = [("sKF, exact", None), ("fKF, exact", None), ("KF, exact", None),
        ("sKF, bank", Q_BANK), ("fKF, bank", Q_BANK), ("KF, bank", Q_BANK)]


def check_best_q(ar_a, beta):
    """BEST_Q against the sweep's saved runs, when all five q of a bank are in results/kf-bank-q-sweep/."""
    for family in ("sKF", "fKF", "KF"):
        paths = [CACHE/f"{family}-bank_grid_q{q:g}_ar{ar_a:g}_beta{beta:g}.pkl" for q in Q_SWEEP]
        if not all(path.exists() for path in paths):
            print(f"   {family} bank: the sweep is not saved here, BEST_Q not checked")
            continue
        means = []
        for path in paths:
            with open(path, "rb") as f:
                means.append(pickle.load(f)["mean"]["mean_db"])
        best = Q_SWEEP[int(np.argmin(means))]
        print(f"   {family} bank: best q of the sweep {best:.3g} ({'as' if best == BEST_Q[(ar_a, beta)][family] else 'NOT as'}"
              f" BEST_Q), {min(means):.2f} dB against {means[0]:.2f} at q = 1e-4")


def base_values(ar_a, beta, v_eta, b_eta, X, D):
    """The base filters' -20 dB eps by notebook 09's search, as tuned_values does for ours: settled points only,
    interpolated (among all points if none settled, and it says so). sKF_batch with chi_gaussian / chi_minorized is
    his sKF_algorithm / sKF_L_algorithm (check 4.1). Saved."""
    def search():
        found = {}
        for name, chi_fn, scale in (("sKF, Gaussian", chi_gaussian, v_eta), ("sKF-L, minorized", chi_minorized, b_eta)):
            floors, settled = sweep("sKF", chi_fn, scale, EPS_GRID, X, D)
            if settled.any():
                value, status = pick(EPS_GRID, floors, settled, TARGET_DB)
            else:                                      # pick needs one settled point; say so instead of failing
                value, status = pick(EPS_GRID, floors, np.ones(len(EPS_GRID), dtype=bool), TARGET_DB)
                status = "NO grid point settled; picked among all of them, " + status
            found[name] = dict(floors=floors, settled=settled, value=float(value), status=status)
        return found
    found = cached(f"search_base_ar{ar_a:g}_beta{beta:g}", search, folder=NEW_CACHE)
    for name, r in found.items():
        print(f"{name}: grid floors " + " ".join(f"{f:.1f}" + ("" if s else "*") for f, s in
                                                 zip(r["floors"], r["settled"])) + "   (* not settled)")
        print(f"   at -20 dB: eps = {r['value']:.4e}, {r['status']}")
    return {name: r["value"] for name, r in found.items()}


def scalar_run(name, params, ar_a, beta, sig, den):
    """One of Augusto's filters or a base filter in one scenario: rest and change runs through run_scalar, summarised by
    summarise (the numbers and curves of ours), and the quartiles of v_t over the change run. Saved."""
    fn = AUGUSTO[name][0] if name in AUGUSTO else BASE[name]
    eps = getattr(params, "epsilon", np.nan)
    key = f"{name.replace(', ', '-').replace(' ', '-')}_{'own' if np.isnan(eps) else f'{eps:.6e}'}_ar{ar_a:g}_beta{beta:g}"

    def compute():
        start = time.time()
        (X_r, D_r), (X_c, D_c) = sig
        mis_r, s_r, _ = run_scalar(fn, params, X_r, D_r)
        mis_c, s_c, v_c = run_scalar(fn, params, X_c, D_c, flip_at=N)
        finite = bool(np.isfinite(mis_r).all() and np.isfinite(mis_c).all() and np.isfinite(s_c).all())
        out = summarise(mis_r, s_r, mis_c, s_c, None, *den)
        v_q = np.percentile(v_c, (25, 50, 75), axis=0)
        out["curves"]["v_q"] = v_q[:, WINDOW].astype(np.float32)
        out["curves"]["v_q_all"] = v_q[:, ::8].astype(np.float32)        # the whole change run, every 8th step
        out["v_at"] = {"N/2": float(np.median(v_c[:, N//2])), "flip": float(np.median(v_c[:, N])),
                       "end": float(np.median(v_c[:, -1]))}
        out.update(name=name, value=eps, q=None, ar_a=ar_a, beta=beta, finite=finite, seconds=time.time() - start)
        return out
    return cached(key, compute, folder=NEW_CACHE)


def show_compare(rows, stat):
    """The table of one scenario, from the mean curves (stat = "mean") or the median curves (stat = "median"):
    show_sweep of kf-bank-q-sweep.ipynb with a wider name column and the setting of each filter."""
    print(f"{'':<32}{' misalignment, ' + stat + ' ':-^72}  {' excess error, ' + stat + ' ':-^38}")
    print(f"{'filter':<20}{'setting':>12}{'floor':>8}{'sett.':>6}{'own+3':>7}{'-17':>7}{'0':>7}{'-10':>7}{'-15':>7}"
          f"{'rec.':>7}{'-20':>7}{'mean':>8}  {'floor':>7}{'-10':>7}{'-15':>7}{'-20':>7}{'mean':>8}")
    for (name, q), r in rows.items():
        m, x = r[stat], r[stat + "_x"]
        setting = f"q {q:.3g}" if q is not None else "own v" if np.isnan(r["value"]) else f"eps {r['value']:.2e}"
        print(f"{name:<20}{setting:>12}{m['floor']:>8.2f}{'yes' if m['settled'] else 'NO':>6}"
              f"{m['reached']:>7d}{m['to_17']:>7d}{m['levels'][0]:>7d}{m['levels'][2]:>7d}{m['levels'][3]:>7d}"
              f"{m['rec']:>7d}{m['back_20']:>7d}{m['mean_db']:>8.2f}  {x['floor']:>7.2f}{x['levels'][2]:>7d}"
              f"{x['levels'][3]:>7d}{x['levels'][4]:>7d}{x['mean_db']:>8.2f}")
    print("   floors and means in dB, everything else in steps; own+3 (to the own floor + 3 dB) and -17 from zero at "
          "rest;\n   0, -10, -15, rec. (back to the own floor + 3 dB) and -20 dB after the flip; -1 = never within "
          "the run;\n   own v = Augusto's rule, no eps")


def scenario_compare(ar_a, beta):
    """Ours (read from results/kf-bank-q-sweep/), the base filters at -20 dB and Augusto's five filters, in one
    scenario; the control and two tables."""
    start = time.time()
    sig = scenario_signals(ar_a, beta)
    (X_r, D_r), (X_c, D_c) = sig
    b_eta, v_eta = noise_of(ar_a, beta)[1], noise_variance(ar_a, beta)
    den = ((clean_output(X_r)**2).mean(axis=0), (clean_output(X_c, change_at=N)**2).mean(axis=0))
    print(f"b_eta = E|eta| = {b_eta:.4e}, v_eta = {v_eta:.4e}; mean of (x_t' theta_t)^2 over the change runs "
          f"{den[1][M:].mean():.4f}, h' R_xx h = {signal_power(ar_a):.4f}")
    tuned = tuned_values(ar_a, beta, b_eta, X_r, D_r)              # ours at -20 dB: given, or from the saved search
    rows = {}
    for name, q in OURS:
        rows[(name, q)] = filter_run(name, np.nan if q else tuned[name.split(", ")[0]], q, ar_a, beta, sig, b_eta, den)
    for family in ("sKF", "fKF", "KF"):
        q = BEST_Q[(ar_a, beta)][family]
        if q != Q_BANK:
            rows[(f"{family}, bank", q)] = filter_run(f"{family}, bank", np.nan, q, ar_a, beta, sig, b_eta, den)
    print(f"ours read [{time.time() - start:.0f} s]")
    for name, eps in base_values(ar_a, beta, v_eta, b_eta, X_r, D_r).items():
        rows[(name, None)] = scalar_run(name, base_params(name, eps, v_eta, b_eta), ar_a, beta, sig, den)
    for name in AUGUSTO:
        rows[(name, None)] = scalar_run(name, augusto_params(name, v_eta, b_eta), ar_a, beta, sig, den)
    print(f"the base filters and Augusto's done [{time.time() - start:.0f} s]")
    print("\ncontrol: ours, read from results/kf-bank-q-sweep/, against kf-bank-q-sweep.ipynb (ours / printed)")
    for (name, q), printed in PRINTED_MEAN[(ar_a, beta)].items():
        m = rows[(name, q)]["mean"]
        line = (f"   {name:<11}{'' if q is None else f'q = {q:.3g}':<11}mean over the record {m['mean_db']:.2f} / "
                f"{printed:.2f}")
        if beta == BETA and (ar_a, name) in PREVIOUS and q in (None, Q_BANK):
            floor, steps, rec, mean = PREVIOUS[(ar_a, name)]
            line += f", floor {m['floor']:.2f} / {floor:.2f}, own+3 {m['reached']} / {steps}, recovery {m['rec']} / {rec}"
        print(line)
    check_best_q(ar_a, beta)
    diverged = [name for (name, q), r in rows.items() if not r.get("finite", True)]
    if diverged:
        print("NOT FINITE in some realisation: " + ", ".join(diverged))
    print()
    show_compare(rows, "mean")
    print()
    show_compare(rows, "median")
    print(f"[{time.time() - start:.0f} s]")
    return rows


print("comparison ready")

## 6. AR($-0.9$), $\beta^* = 0.2$

The scenario of the earlier notebooks. Control: ours must give the numbers printed by `kf-bank-q-sweep.ipynb`.

In [ ]:
results[(-0.9, BETA)] = scenario_compare(-0.9, BETA)

## 7. White input, $\beta^* = 0.2$

The same control.

In [ ]:
results[(0.0, BETA)] = scenario_compare(0.0, BETA)

## 8. AR($-0.9$), Gaussian noise ($\beta^* = 2$)

Ours at $-20$ dB use the values found by the search of `kf-bank-q-sweep.ipynb` (read from its saved search; none of its settled grid points reaches $-20$ dB here, so they sit at higher floors).

In [ ]:
results[(-0.9, BETA_GAUSS)] = scenario_compare(-0.9, BETA_GAUSS)

## 9. White input, Gaussian noise ($\beta^* = 2$)

As above.

In [ ]:
results[(0.0, BETA_GAUSS)] = scenario_compare(0.0, BETA_GAUSS)

## 10. Figures

Figures 1 and 3 run from 0.25 s before the change to 2.5 s after it with both inputs, the whole window kept by
`summarise`; the earlier notebooks stopped at 0.6 s with white input, but Augusto's filters and the base filters take
longer.

**Figure 1.** Augusto's filters beside the same filter with $\varepsilon$ fixed at $-20$ dB (grey, dashed), across the
change: misalignment, median over the 20 realisations and its 25-75 % band. Rows: the four scenarios. Left: the Gaussian
likelihood (his `sKF_algorithm` at $-20$ dB, AWU-sKF v1, v2, GISV-sKF); right: the minorized Laplacian one (his
`sKF_L_algorithm` at $-20$ dB, AWU-sKF-L v1, v2). Colour = his rule (AWU v1 yellow, AWU v2 magenta, GISV green), the
same on both sides. AWU-sKF v2 is drawn wide under GISV-sKF, and AWU-sKF-L v2 under AWU-sKF-L v1, which can hide them.
Grey: $-20$ dB; black: the change.

**Figure 2.** $v_t$ of the same filters over the whole change run (at rest, then the flip at 0), median and 25-75 %
band, log scale.

**Figure 3.** Augusto's three Gaussian filters against our six, misalignment (left) and excess error (right), median.
Ours as in `kf-bank-q-sweep.ipynb`: colour = family (sKF orange, fKF aqua, KF blue), solid = bank at $q = 10^{-4}$,
dashed = single filter at $-20$ dB. The AWU-sKF-L filters are left out; after the flip they stay near $+6$ dB
(Figure 1).

**Figure 4.** Every filter of the tables, from the mean curves: the mean over the record (the draft's metric, top) and
the floor at rest (bottom), one column per scenario. The "best q" rows are each bank at its best $q$, printed beside the
dot. Every plotted number is also in the tables of sections 6-9.

In [ ]:
# === copied from kf-bank-q-sweep.ipynb section 10 (commit 56108a2): the colours of the families and the
#     axis helpers, NOT edited ===
FAMILY_COLOUR = {"KF": "#2a78d6", "sKF": "#eb6834", "fKF": "#1baf7a"}
SINGLE_GREY, TARGET_GREY, GRID_GREY, INK = "#52514e", "#a3a29b", "#e1e0d9", "#0b0b0b"
SCENARIOS = [(-0.9, BETA), (0.0, BETA), (-0.9, BETA_GAUSS), (0.0, BETA_GAUSS)]
T_MAX = {-0.9: 2.5, 0.0: 0.6}                  # seconds after the change shown, as in the earlier notebooks


def scenario_label(ar_a, beta):
    return f"{'AR(-0.9)' if ar_a else 'white'}, beta* = {beta:g}"


def window_time(curve):
    # time from the change [s] of the kept window of the change run
    return (np.arange(curve.shape[-1]) + WINDOW.start - N)/FS


def style_axes(ax, t_max, ylim):
    ax.axvline(0, color=INK, lw=0.8)                          # the change
    ax.axhline(TARGET_DB, color=TARGET_GREY, lw=0.8)          # -20 dB
    ax.set(xlim=(-0.25, t_max), ylim=ylim)
    ax.grid(color=GRID_GREY, lw=0.6)
    ax.set_axisbelow(True)


def db_range(curves, top=8.0):
    # y range of a row: from below the lowest median (5 dB steps) to top
    low = min(float(np.nanmin(10*np.log10(np.maximum(c, 1e-30)))) for c in curves)
    return (5*np.floor(low/5) - 2, top)
# === end of the copied block ===


T_MAX = {-0.9: 2.5, 0.0: 2.5}       # here the whole kept window with both inputs: with white input Augusto's filters and
                                    # the base filters take longer than the 0.6 s of kf-bank-q-sweep.ipynb

# Augusto's three rules take slots 4-6 of the categorical palette whose slots 1-3 are the families above, and the base
# filters the grey of the single filters. Checked with the dataviz skill's validator on white: slots 1-6 pass the
# adjacent CVD and normal-vision checks; yellow, magenta and aqua are below 3:1 contrast on white, so every plotted
# number is also in the tables of sections 6-9.
RULE_COLOUR = {"AWU v1": "#eda100", "AWU v2": "#e87ba4", "GISV": "#008300"}
LOOK = {"sKF, exact": (FAMILY_COLOUR["sKF"], "--"), "fKF, exact": (FAMILY_COLOUR["fKF"], "--"),
        "KF, exact": (FAMILY_COLOUR["KF"], "--"), "sKF, bank": (FAMILY_COLOUR["sKF"], "-"),
        "fKF, bank": (FAMILY_COLOUR["fKF"], "-"), "KF, bank": (FAMILY_COLOUR["KF"], "-"),
        "sKF, Gaussian": (SINGLE_GREY, "--"), "sKF-L, minorized": (SINGLE_GREY, "--"),
        "AWU-sKF v1": (RULE_COLOUR["AWU v1"], "-"), "AWU-sKF v2": (RULE_COLOUR["AWU v2"], "-"),
        "GISV-sKF": (RULE_COLOUR["GISV"], "-"), "AWU-sKF-L v1": (RULE_COLOUR["AWU v1"], "-"),
        "AWU-sKF-L v2": (RULE_COLOUR["AWU v2"], "-")}
WIDTH = {"AWU-sKF v2": 3.2, "AWU-sKF-L v2": 3.2}      # drawn wide, under GISV-sKF / AWU-sKF-L v1, which can hide them
GROUPS = {"Gaussian likelihood": ["sKF, Gaussian", "AWU-sKF v1", "AWU-sKF v2", "GISV-sKF"],
          "minorized Laplacian likelihood": ["sKF-L, minorized", "AWU-sKF-L v2", "AWU-sKF-L v1"]}
SUMMARY_ROWS = [("sKF, exact", None), ("fKF, exact", None), ("KF, exact", None), ("sKF, bank", Q_BANK),
                ("fKF, bank", Q_BANK), ("KF, bank", Q_BANK), ("sKF, bank", "best"), ("fKF, bank", "best"),
                ("KF, bank", "best"), ("sKF, Gaussian", None), ("AWU-sKF v1", None), ("AWU-sKF v2", None),
                ("GISV-sKF", None), ("sKF-L, minorized", None), ("AWU-sKF-L v1", None), ("AWU-sKF-L v2", None)]


def label_of(name, q=None):
    if q is not None:
        return f"{name}, q = {q:.3g}"
    return f"{name}, -20 dB" if name.endswith("exact") or name in BASE else name


def figure_rules(results):
    """Figure 1: Augusto's filters and the same filter with eps fixed at -20 dB, misalignment across the change."""
    fig, axes = plt.subplots(4, 2, figsize=(15, 17), constrained_layout=True, sharey="row")
    for row, (ar_a, beta) in enumerate(SCENARIOS):
        rows = results[(ar_a, beta)]
        t = window_time(rows[("GISV-sKF", None)]["curves"]["mis_q"][1])
        keep = t <= T_MAX[ar_a]
        ylim = db_range([rows[(name, None)]["curves"]["mis_q"][1][keep] for names in GROUPS.values() for name in names])
        for col, (group, names) in enumerate(GROUPS.items()):
            ax = axes[row, col]
            for name in names:
                colour, style = LOOK[name]
                lo, mid, hi = rows[(name, None)]["curves"]["mis_q"]
                ax.fill_between(t[keep], 10*np.log10(lo[keep]), 10*np.log10(hi[keep]), color=colour, alpha=0.14, lw=0)
                ax.plot(t[keep], 10*np.log10(mid[keep]), color=colour, ls=style, lw=WIDTH.get(name, 1.5),
                        label=label_of(name))
            style_axes(ax, T_MAX[ar_a], ylim)
            ax.set_title(f"{scenario_label(ar_a, beta)}: {group}", fontsize=10)
            ax.set_xlabel("time from the change [s]")
            if row == 0:
                ax.legend(fontsize=8, loc="upper right")
        axes[row, 0].set_ylabel("misalignment, median [dB]")
    fig.suptitle("Augusto's filters beside the same filter with eps fixed at -20 dB: misalignment, median and "
                 "25-75 % band")
    plt.show()


def figure_v(results):
    """Figure 2: v_t of the same filters over the whole change run."""
    fig, axes = plt.subplots(4, 2, figsize=(15, 17), constrained_layout=True, sharey="row")
    t = (np.arange(0, 2*N, 8) - N)/FS                       # the steps kept in v_q_all, time from the change
    for row, (ar_a, beta) in enumerate(SCENARIOS):
        rows = results[(ar_a, beta)]
        for col, (group, names) in enumerate(GROUPS.items()):
            ax = axes[row, col]
            for name in names:
                colour, style = LOOK[name]
                lo, mid, hi = rows[(name, None)]["curves"]["v_q_all"]
                ax.fill_between(t, lo, hi, color=colour, alpha=0.14, lw=0)
                ax.plot(t, mid, color=colour, ls=style, lw=WIDTH.get(name, 1.5), label=label_of(name))
            ax.set_yscale("log")
            ax.axvline(0, color=INK, lw=0.8)
            ax.set_xlim(t[0], t[-1])
            ax.grid(color=GRID_GREY, lw=0.6)
            ax.set_axisbelow(True)
            ax.set_title(f"{scenario_label(ar_a, beta)}: {group}", fontsize=10)
            ax.set_xlabel("time from the change [s]")
            if row == 0:
                ax.legend(fontsize=8, loc="upper right")
        axes[row, 0].set_ylabel("v_t, median")
    fig.suptitle("The variance v_t of the same filters over the whole change run, median and 25-75 % band")
    plt.show()


def figure_vs_ours(results):
    """Figure 3: Augusto's three Gaussian filters against our six, misalignment and excess error, median."""
    picks = OURS + [("AWU-sKF v1", None), ("AWU-sKF v2", None), ("GISV-sKF", None)]
    fig, axes = plt.subplots(4, 2, figsize=(15, 17), constrained_layout=True, sharey="row")
    for row, (ar_a, beta) in enumerate(SCENARIOS):
        rows = results[(ar_a, beta)]
        t = window_time(rows[picks[0]]["curves"]["mis_q"][1])
        keep = t <= T_MAX[ar_a]
        ylim = db_range([rows[key]["curves"][f"{metric}_q"][1][keep] for key in picks for metric in ("mis", "exc")])
        for col, metric in enumerate(("mis", "exc")):
            ax = axes[row, col]
            for name, q in picks:
                colour, style = LOOK[name]
                mid = rows[(name, q)]["curves"][f"{metric}_q"][1]
                ax.plot(t[keep], 10*np.log10(mid[keep]), color=colour, ls=style, lw=WIDTH.get(name, 1.5),
                        label=label_of(name, q))
            style_axes(ax, T_MAX[ar_a], ylim)
            ax.set_title(f"{scenario_label(ar_a, beta)}: " + ("misalignment" if metric == "mis" else "excess error"),
                         fontsize=10)
            ax.set_xlabel("time from the change [s]")
        axes[row, 0].set_ylabel("median [dB]")
    fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside lower center", ncol=5, fontsize=9)
    fig.suptitle("Augusto's Gaussian filters against ours (banks at q = 1e-4, single filters at -20 dB): median")
    plt.show()


def figure_summary(results):
    """Figure 4: the mean over the record and the floor at rest of every filter, from the mean curves."""
    fig, axes = plt.subplots(2, 4, figsize=(19, 10), constrained_layout=True, sharey=True)
    y = np.arange(len(SUMMARY_ROWS))
    for col, (ar_a, beta) in enumerate(SCENARIOS):
        rows = results[(ar_a, beta)]
        for row, (key, label) in enumerate((("mean_db", "mean over the record [dB]"), ("floor", "floor at rest [dB]"))):
            ax = axes[row, col]
            for i, (name, q) in enumerate(SUMMARY_ROWS):
                q_used = BEST_Q[(ar_a, beta)][name.split(", ")[0]] if q == "best" else q
                value = rows[(name, q_used)]["mean"][key]
                ax.plot(value, i, "o", color=LOOK[name][0], ms=7, mec="white", mew=1.2)
                if q == "best":
                    ax.annotate(f"q = {q_used:.3g}", (value, i), xytext=(7, 0), textcoords="offset points",
                                va="center", fontsize=7, color=SINGLE_GREY)
            if key == "floor":
                ax.axvline(TARGET_DB, color=TARGET_GREY, lw=0.8)
            ax.grid(color=GRID_GREY, lw=0.6)
            ax.set_axisbelow(True)
            ax.set_xlabel(label)
            if row == 0:
                ax.set_title(scenario_label(ar_a, beta), fontsize=10)
    axes[0, 0].set_yticks(y, [label_of(name, None if q is None else q) if q != "best" else f"{name}, best q"
                              for name, q in SUMMARY_ROWS])
    for ax in axes.flat:
        ax.tick_params(axis="y", labelsize=8)
    axes[0, 0].invert_yaxis()
    fig.suptitle("Every filter, from the mean curves: the mean over the record (the draft's metric) and the floor at rest")
    plt.show()


figure_rules(results)

In [ ]:
figure_v(results)

In [ ]:
figure_vs_ours(results)

In [ ]:
figure_summary(results)

## 11. Why the AWU-sKF-L filters freeze

With the minorized likelihood, the predicted variance of $e_t$ in the AWU rules is
$J_t = b_\eta\lvert e_t\rvert + v\lVert\boldsymbol{x}_t\rVert^2$, which is small whenever $\lvert e_t\rvert$ is small.

- **AWU-sKF-L v2.** While $v\lVert\boldsymbol{x}_t\rVert^2 \ll b_\eta\lvert e_t\rvert$, one step (its inflation, then the
  update) multiplies $v$ by about
  $$1 + \frac{v\lVert\boldsymbol{x}_t\rVert^2}{M b_\eta}\Big(\frac{1}{b_\eta} - \frac{1}{\lvert e_t\rvert}\Big),$$
  which is below 1 whenever $\lvert e_t\rvert < b_\eta$, and the more so the smaller $\lvert e_t\rvert$ (down to the
  factor $1 - 1/M$ at $e_t = 0$). At rest $e_t$ is mostly noise, and the noise is below $b_\eta = \mathrm{E}\lvert\eta_t\rvert$
  on most steps (printed below), so $v$ keeps falling. After the flip $\lvert e_t\rvert \gg b_\eta$ and $v$ grows, but
  only by the relative amount $v\lVert\boldsymbol{x}_t\rVert^2/(M b_\eta^2) \approx v/b_\eta^2$ per step, which is tiny once
  $v$ is small.
- **AWU-sKF-L v1** shrinks faster still: as $\lvert e_t\rvert \to 0$, $J_t \to v\lVert\boldsymbol{x}_t\rVert^2$ and its factor
  $1 + v\lVert\boldsymbol{x}_t\rVert^2(e_t^2 - J_t)/J_t^2$ goes to 0, so one small error can remove almost all of $v$.
- With $v$ that small the filter can hardly move: the minorized step changes $\boldsymbol{w}$ by at most
  $\lVert\boldsymbol{x}_t\rVert\tilde v_t/b_\eta$ (its gain $\tilde v_t e_t/(b_\eta\lvert e_t\rvert + \tilde v_t\lVert\boldsymbol{x}_t\rVert^2)$ is at
  most $\tilde v_t/b_\eta$), and the flip moves $\boldsymbol{\theta}$ by $\lVert 2\boldsymbol{h}_o\rVert = 2$. The last column
  below is the number of steps that would take with $v$ held at its value at the flip and $\lVert\boldsymbol{x}_t\rVert^2 = M$.

The Gaussian AWU rules do not have this problem: their $J_t \ge v_\eta$, so a small $e_t$ multiplies $v$ by no less than
$v_\eta/J_t$ (v1) or not at all (v2).

In [ ]:
print("P(|eta_t| < b_eta) at rest, the share of steps with |e_t| < b_eta (the ratio b_eta / scale depends on beta* only):")
for beta in (BETA, BETA_GAUSS):
    scale_gg, b_eta = noise_of(-0.9, beta)
    print(f"   beta* = {beta:g}: {gennorm.cdf(b_eta/scale_gg, beta) - gennorm.cdf(-b_eta/scale_gg, beta):.2f}")
print()
print(f"{'scenario':<23}{'filter':<18}{'v at N/2':>10}{'at flip':>10}{'peak/flip':>11}{'at end':>10}{'end/flip':>10}"
      f"{'steps to undo the flip':>24}")
for ar_a, beta in SCENARIOS:
    b_eta = noise_of(ar_a, beta)[1]
    for name in list(BASE) + list(AUGUSTO):
        r = results[(ar_a, beta)][(name, None)]
        v_at, peak = r["v_at"], float(np.max(r["curves"]["v_q_all"][1][N//8:]))    # v_q_all keeps every 8th step
        bound = f"{2*b_eta/(np.sqrt(M)*v_at['flip']):.1e}" if name in ("AWU-sKF-L v1", "AWU-sKF-L v2") else ""
        print(f"{scenario_label(ar_a, beta):<23}{name:<18}{v_at['N/2']:>10.1e}{v_at['flip']:>10.1e}"
              f"{peak/v_at['flip']:>11.1f}{v_at['end']:>10.1e}{v_at['end']/v_at['flip']:>10.2f}{bound:>24}")
print("   v: median over the 20 realisations of the change run, before step N/2, before the flip (step N) and at the "
      "last step;\n   peak/flip: the largest median v after the flip over the one at the flip; steps to undo the flip: "
      "2 b_eta / (sqrt(M) v at the flip),\n   for the AWU-sKF-L filters only")

## 12. Findings

**Checks: all pass.**
1. His `sKF_algorithm` and `sKF_L_algorithm` are `sKF_batch` with the Gaussian and the minorized $\chi$, and `run_scalar`
   forms the misalignment and $s_t$ as `sKF_batch` does: to $3 \times 10^{-15}$ and $6 \times 10^{-16}$ (4.1).
2. GISV-sKF adds $\lVert\boldsymbol{w}_t - \boldsymbol{w}_{t-1}\rVert^2/M$ to $v$, to $2 \times 10^{-16}$: it is the estimator of
   the draft's Remark 5. Over the last 1 000 steps of that check (500 to 1 500 steps after its flip) the amount is about
   0.1 % of $v_t$; the draft's Remark 5 has it below 1 % (4.2).
3. Control: every number of ours read from `results/kf-bank-q-sweep/` is the one `kf-bank-q-sweep.ipynb` printed, and
   `BEST_Q` is the best $q$ of the saved sweep in all 12 cases.
4. Cross-check: his minorized sKF-L, tuned to $-20$ dB here ($\varepsilon = 2.64 \times 10^{-6}$), takes 4 357 steps from
   zero and 7 340 to recover with AR($-0.9$) and $\beta^* = 0.2$: notebook 09's minorized filter to the step (the
   control of `clipped-matched.ipynb`).

**1. AWU-sKF-L v1 and v2 do not follow the change, in any of our scenarios nor in Augusto's own setup.**
- After the flip both stay near $+6$ dB, the old response ($\lVert 2\boldsymbol{h}_o\rVert^2$), until the end of the run,
  96 000 steps later. Their means over the record, $+0.1$ to $+3.0$ dB, are worse than not adapting at all
  ($\boldsymbol{w} = \boldsymbol{0}$ gives 0 dB). AWU-sKF-L v1 does not even converge at rest (floors $-0.2$ to $-3.5$ dB);
  v2 does ($-8.6$ to $-46.7$ dB).
- In his setup (4.3) the same, with both of his noises: $-21$ dB (v1) and $-53$ dB (v2) before the flip, $+5.9$ to
  $+6.0$ dB from 1 000 steps after it to the end.
- Why (section 11): with $v_\eta$ replaced by $b_\eta\lvert e_t\rvert$, a small error makes the predicted variance $J_t$
  small, and the rule shrinks $v$ whenever $\lvert e_t\rvert < b_\eta$: at rest, where $e_t$ is mostly noise, on about
  81 % of the steps with $\beta^* = 0.2$ and 58 % with Gaussian noise. So $v$ falls through the whole rest run, to $1$-$5 \times 10^{-9}$ (v1) and
  $5 \times 10^{-9}$ to $6 \times 10^{-7}$ (v2) at the flip, and the flip does not raise it: its largest value after the
  flip is at most 1.1 times its value at the flip (Figure 2). With that $v$, undoing the flip would take
  $1.2 \times 10^5$ to $2 \times 10^7$ steps.
- So this comes from the two rules as written at `6a27d96`, not from our scenario. Worth passing on to Augusto.

**2. AWU-sKF v2 and GISV-sKF are one filter in practice.** They add the same amount to $v$, one step apart. Their curves
lie on top of each other (Figures 1-3), and their means over the record differ by 0.6 dB at most.

**3. Against the same filter with $\varepsilon$ fixed at $-20$ dB, the Gaussian rules help; AWU-sKF v1 only with white
input and Gaussian noise.** Mean over the record [dB], mean curves:

| scenario | sKF, Gaussian, $-20$ dB | AWU-sKF v1 | AWU-sKF v2 | GISV-sKF | sKF-L, minorized, $-20$ dB | AWU-sKF-L v1 | AWU-sKF-L v2 |
|---|---|---|---|---|---|---|---|
| AR, $\beta^* = 0.2$ | $-7.42$* | $-1.81$ | $-8.32$ | $-8.37$ | $-13.97$ | $+0.34$ | $+2.66$ |
| white, $\beta^* = 0.2$ | $-14.52$ | $-11.36$ | $-14.71$ | $-15.27$ | $-17.29$ | $+1.40$ | $+3.01$ |
| AR, $\beta^* = 2$ | $-9.19$* | $-7.50$ | $-11.77$ | $-11.80$ | $-6.91$* | $+0.06$ | $+2.38$ |
| white, $\beta^* = 2$ | $-14.49$ | $-19.65$ | $-16.22$ | $-16.21$ | $-11.65$ | $+0.09$ | $+2.81$ |

\* no settled grid point reaches $-20$ dB; the search took the deepest settled one ($-19.7$, $-17.3$ and $-18.2$ dB).

- AWU-sKF v2 and GISV-sKF beat their base in every scenario, by 0.2 to 2.6 dB. Their $v$ keeps drifting down at rest
  (Figure 2), so their floors are below the base's in three scenarios of four ($-22$ to $-27$ dB, against $-17.3$ to
  $-20$). At the flip their $v$ jumps 25 to 67 times, where the base's does not move, so they reach 0 and $-10$ dB
  sooner than the base in every scenario. To $-15$ dB they are sooner in two scenarios of four: they are slower with
  AR input and $\beta^* = 0.2$ (92 260 and 95 137 steps against 57 986) and with white input and $\beta^* = 2$ (6 114 and
  6 129 against 5 646).
- AWU-sKF v1 beats its base only with white input and Gaussian noise, by 5.2 dB: $-19.65$ dB, the best mean of all the
  filters in that scenario, with a floor of $-31.3$ dB that is still falling at the end of the rest run. In the three
  other scenarios it is worse than its base, by 1.7 to 5.6 dB, and with AR input and $\beta^* = 0.2$ it does not
  converge (floor $-2.7$ dB). Before the flip its $v$ is 7 to 41 times below that of AWU-sKF v2, and at the flip it
  jumps 270 to 6 800 times.

**4. Against our filters: behind them with heavy-tailed noise; ahead of our single filters with Gaussian noise, where
only their likelihood is the right one.** Mean over the record [dB], mean curves:

| scenario | best of Augusto's | sKF / fKF / KF at $-20$ dB | sKF / fKF / KF bank, $q = 10^{-4}$ | banks at their best $q$, if not $10^{-4}$ |
|---|---|---|---|---|
| AR, $\beta^* = 0.2$ | $-8.37$ (GISV-sKF) | $-14.08$ / $-13.45$ / $-15.63$ | $-13.89$ / $-11.92$ / $\mathbf{-20.55}$ | sKF $-16.52$ ($q = 0.1$), KF $\mathbf{-20.57}$ ($10^{-3}$) |
| white, $\beta^* = 0.2$ | $-15.27$ (GISV-sKF) | $-17.22$ / $-16.67$ / $-17.65$ | $-22.64$ / $-23.01$ / $\mathbf{-23.92}$ | KF $\mathbf{-23.94}$ ($10^{-3}$) |
| AR, $\beta^* = 2$ | $-11.80$ (GISV-sKF) | $-6.77$* / $-8.00$* / $-6.79$* | $-8.68$ / $-8.00$ / $\mathbf{-14.09}$ | sKF $-10.53$ ($10^{-2}$), fKF $-9.46$ ($1/3$) |
| white, $\beta^* = 2$ | $\mathbf{-19.65}$ (AWU-sKF v1) | $-11.56$ / $-10.27$ / $-12.06$ | $-18.55$ / $-17.09$ / $-18.89$ | (all at $10^{-4}$) |

\* not at $-20$ dB: no settled grid point reaches it (`kf-bank-q-sweep.ipynb`, section 8). Bold: the best of the row.

- **Heavy-tailed noise ($\beta^* = 0.2$): every filter of ours is ahead of every filter of Augusto's.** His best,
  GISV-sKF, is 5.7 dB (AR) and 2.0 dB (white) behind the exact sKF, and 12.2 and 8.7 dB behind the KF bank. Most of it
  is the Gaussian likelihood: the Gaussian sKF tuned to $-20$ dB is 6.7 and 2.7 dB behind the exact sKF, and the rules
  give back about 1 dB of that.
- **Gaussian noise ($\beta^* = 2$), where his likelihood is the right one and ours is not.** Our filters keep the
  Laplacian likelihood under Gaussian noise, so that only the noise changes (`kf-bank-q-sweep.ipynb`). AWU-sKF v2 and
  GISV-sKF beat our single filters by 3.8 to 6.0 dB, and with AR input also the sKF and fKF banks at their best $q$. The
  KF bank stays ahead with AR input ($-14.09$ against $-11.80$), and all three banks stay ahead with white input
  ($-17.1$ to $-18.9$ against $-16.2$). AWU-sKF v1 with white input is 0.8 dB ahead of the best bank. Part of this is
  the likelihood: with white input the Gaussian sKF at $-20$ dB is 2.9 dB ahead of our exact sKF at the same floor
  ($-14.49$ against $-11.56$).
- **With AR input the KF bank is still the best filter in both noises**, ahead of the best of Augusto's by 12.2 dB
  ($\beta^* = 0.2$) and 2.3 dB ($\beta^* = 2$).

**5. Excess error.** The same order as the misalignment. With AR input and $\beta^* = 0.2$, AWU-sKF v2 and GISV-sKF leave
their error where the input has little power, as the KF does: their excess floors are about 6 dB below their
misalignment floors ($-24.8$ and $-25.6$ against $-18.7$ and $-19.1$ dB; the KF $-5.2$ dB, the exact sKF $+1.0$).

**Not done.**
- Augusto's rules on our exact Laplacian $\chi$. GISV-sKF with that $\chi$ is the "eq. (76) of [5]" entry of the draft's
  Remark 6, which reports that with the Laplacian likelihood the estimate tends to zero. Not in the scope chosen.
- Our filters in Augusto's setup: only his five filters ran there, as a check (4.3).
- The $q$-switched sKF bank of `skf-bank-q-switch.ipynb`, and the NLMS filters of his notebook.

In [ ]:
new_runs = sum(r["seconds"] for rows in results.values() for (name, q), r in rows.items()
               if name in AUGUSTO or name in BASE)
print(f"whole notebook: {(time.time() - NOTEBOOK_START)/60:.1f} min; the runs of Augusto's and the base filters took "
      f"{new_runs/60:.1f} min when they were run")